In [4]:
r"""
lulc_dataset.py
===============
Dataset PyTorch para predicao de transicao para soja no Cerrado.
Conforme Contrato do Dataset GeoFM v2.

Tarefa:
  - Positivo (label=1): pixel com pastagem RAPIDA antes de soja (d <= 4)
  - Negativo (label=0): pixel com pastagem LONGA antes de soja  (d >  4)

USO:
  df = pd.read_csv(r"...\indice_treino.csv")
  ds = LULCTransitionDataset(df)
  verificar_dataset(ds, n=10)
"""

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import torch
from torch.utils.data import Dataset

# ============================================================
# CONFIGURACAO GLOBAL
# ============================================================
DATA_DIR      = r"D:\Projetos\Cerrado\LULC"
PATTERN       = "brazil_coverage_{ano}_Cerrado.tif"
YEARS         = list(range(1985, 2025))
NODATA_IN     = 255
NODATA_OUT    = 0
PATCH_N       = 7
MAX_SERIE_LEN = len(YEARS) - 1   # = 39
PATCH_FRAMES  = 5

# ============================================================
# FUNCOES AUXILIARES DO DATASET
# ============================================================

def dataset_ler_pixel(year, row, col, data_dir, pattern):
    path = os.path.join(data_dir, pattern.format(ano=year))
    with rasterio.open(path) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def dataset_ler_patch(year, row, col, n, data_dir, pattern):
    half = n // 2
    with rasterio.open(os.path.join(data_dir, pattern.format(ano=year))) as ds:
        H, W = ds.height, ds.width
        col0 = min(max(0, col - half), W - n)
        row0 = min(max(0, row - half), H - n)
        arr  = ds.read(1, window=Window(col0, row0, n, n), out_dtype="uint8")
    return np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)

# ============================================================
# DATASET PYTORCH
# ============================================================

class LULCTransitionDataset(Dataset):
    def __init__(
        self,
        indice_df,
        data_dir=DATA_DIR,
        pattern=PATTERN,
        years=YEARS,
        nodata_in=NODATA_IN,
        nodata_out=NODATA_OUT,
        patch_n=PATCH_N,
        max_serie_len=MAX_SERIE_LEN,
        patch_frames=PATCH_FRAMES,
    ):
        self.df            = indice_df.reset_index(drop=True)
        self.data_dir      = data_dir
        self.pattern       = pattern
        self.years         = years
        self.nodata_in     = nodata_in
        self.nodata_out    = nodata_out
        self.patch_n       = patch_n
        self.max_serie_len = max_serie_len
        self.patch_frames  = patch_frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r     = self.df.iloc[idx]
        row   = int(r["row"])
        col   = int(r["col"])
        T     = int(r["T"])
        label = int(r["label"])

        # 1. Serie temporal (1985..T-1) com padding a esquerda
        anos_serie = [y for y in self.years if y < T]
        serie_raw  = np.array(
            [dataset_ler_pixel(y, row, col, self.data_dir, self.pattern)
             for y in anos_serie],
            dtype=np.float32
        )
        serie_raw = np.where(
            serie_raw == self.nodata_in, self.nodata_out, serie_raw
        ).astype(np.float32)

        serie_len = len(serie_raw)
        serie     = np.zeros(self.max_serie_len, dtype=np.float32)
        serie[self.max_serie_len - serie_len:] = serie_raw

        # 2. Patch espacial (5 x N x N) com padding a esquerda
        anos_patch = [y for y in self.years if y < T][-self.patch_frames:]
        frames = [
            dataset_ler_patch(y, row, col, self.patch_n, self.data_dir, self.pattern)
            for y in anos_patch
        ]
        patch_len = len(frames)
        if patch_len < self.patch_frames:
            pad    = [np.zeros((self.patch_n, self.patch_n), dtype=np.uint8)] * (self.patch_frames - patch_len)
            frames = pad + frames
        patch = np.stack(frames, axis=0).astype(np.float32)

        # 3. Features auxiliares
        has_21    = float(np.sum(serie_raw == 21)) / serie_len
        anos_past = 0
        for v in reversed(serie_raw):
            if v == 15: anos_past += 1
            else: break
        cl_tm1   = float(serie_raw[-1])
        features = np.array([has_21, float(anos_past), cl_tm1], dtype=np.float32)

        return {
            "serie"    : torch.tensor(serie,     dtype=torch.float32),
            "serie_len": torch.tensor(serie_len, dtype=torch.long),
            "patch"    : torch.tensor(patch,     dtype=torch.float32),
            "patch_len": torch.tensor(patch_len, dtype=torch.long),
            "features" : torch.tensor(features,  dtype=torch.float32),
            "label"    : torch.tensor(label,     dtype=torch.long),
            "meta"     : {"row": row, "col": col, "T": T},
        }

# ============================================================
# VERIFICACAO
# ============================================================

def verificar_dataset(dataset, n=10, seed=42):
    import random
    random.seed(seed)
    indices    = random.sample(range(len(dataset)), min(n, len(dataset)))
    SEP        = "=" * 70
    resultados = []

    print(f"\n{SEP}\nVERIFICACAO DO __getitem__\n{SEP}")

    for i, idx in enumerate(indices):
        s        = dataset[idx]
        r        = dataset.df.iloc[idx]
        T        = int(r["T"])
        row      = int(r["row"])
        col      = int(r["col"])
        lbl_esp  = int(r["label"])

        serie     = s["serie"].numpy()
        serie_len = int(s["serie_len"].item())
        patch     = s["patch"].numpy()
        patch_len = int(s["patch_len"].item())
        feats     = s["features"].numpy()
        label     = int(s["label"].item())

        anos_esp = [y for y in dataset.years if y < T]
        len_esp  = len(anos_esp)
        n_pad_s  = dataset.max_serie_len - serie_len
        n_pad_p  = dataset.patch_frames  - patch_len

        print(f"\nAMOSTRA {i+1:02d} | idx={idx} | row={row}, col={col} | T={T} | label={label}")
        print("─" * 70)

        c1 = (serie.shape == (dataset.max_serie_len,) and serie_len == len_esp
              and anos_esp[-1] == T - 1
              and (np.all(serie[:n_pad_s] == 0) if n_pad_s > 0 else True))
        print(f"[C1] shape={serie.shape} | len={serie_len} (esp {len_esp}) | "
              f"ultimo_ano={anos_esp[-1]} | padding_ok={np.all(serie[:n_pad_s]==0) if n_pad_s>0 else True}")
        print(f"     {'OK' if c1 else 'FALHA'}")

        c2 = not np.any(serie == 255)
        print(f"[C2] Sem nodata=255 -> {'OK' if c2 else 'FALHA'}")

        c3 = (patch.shape == (dataset.patch_frames, dataset.patch_n, dataset.patch_n)
              and not np.any(patch == 255)
              and (np.all(patch[:n_pad_p] == 0) if n_pad_p > 0 else True))
        print(f"[C3] patch.shape={patch.shape} | patch_len={patch_len} -> {'OK' if c3 else 'FALHA'}")

        c4 = (label == lbl_esp)
        print(f"[C4] label={label} (esp {lbl_esp}) -> {'OK' if c4 else 'FALHA'}")

        has_21, anos_past, cl_tm1 = feats
        c5 = (0.0 <= has_21 <= 1.0 and anos_past >= 0 and cl_tm1 >= 0)
        print(f"[C5] has_21={has_21:.3f} | anos_past={int(anos_past)} | "
              f"cl_tm1={int(cl_tm1)} -> {'OK' if c5 else 'FALHA'}")

        passou = c1 and c2 and c3 and c4 and c5
        print(f"RESULTADO: {'PASSOU' if passou else 'FALHA'}")
        resultados.append(passou)

    print(f"\n{SEP}")
    print(f"RESUMO: {sum(resultados)}/{len(resultados)} passaram")
    print("Dataset consistente com o contrato." if all(resultados) else "Existem falhas.")
    print(SEP)
    return all(resultados)

# ============================================================
# EXECUCAO
# ============================================================
INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"

df_treino = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))
df_val    = pd.read_csv(os.path.join(INDEX_DIR, "indice_val.csv"))
df_teste  = pd.read_csv(os.path.join(INDEX_DIR, "indice_teste.csv"))

ds_treino = LULCTransitionDataset(df_treino)
ds_val    = LULCTransitionDataset(df_val)
ds_teste  = LULCTransitionDataset(df_teste)

print(f"Dataset treino : {len(ds_treino)} amostras")
print(f"Dataset val    : {len(ds_val)} amostras")
print(f"Dataset teste  : {len(ds_teste)} amostras")

verificar_dataset(ds_treino, n=10)

from torch.utils.data import DataLoader
loader = DataLoader(ds_treino, batch_size=4, shuffle=True, num_workers=0)
batch  = next(iter(loader))
print(f"\nExemplo de batch:")
print(f"  serie     : {batch['serie'].shape}")
print(f"  serie_len : {batch['serie_len'].tolist()}")
print(f"  patch     : {batch['patch'].shape}")
print(f"  features  : {batch['features'].shape}")
print(f"  label     : {batch['label'].tolist()}")

Dataset treino : 10000 amostras
Dataset val    : 1000 amostras
Dataset teste  : 420 amostras

VERIFICACAO DO __getitem__

AMOSTRA 01 | idx=1824 | row=38882, col=15060 | T=2008 | label=0
──────────────────────────────────────────────────────────────────────
[C1] shape=(39,) | len=23 (esp 23) | ultimo_ano=2007 | padding_ok=True
     OK
[C2] Sem nodata=255 -> OK
[C3] patch.shape=(5, 7, 7) | patch_len=5 -> OK
[C4] label=0 (esp 0) -> OK
[C5] has_21=0.000 | anos_past=0 | cl_tm1=4 -> OK
RESULTADO: PASSOU

AMOSTRA 02 | idx=409 | row=21933, col=50108 | T=2002 | label=1
──────────────────────────────────────────────────────────────────────
[C1] shape=(39,) | len=17 (esp 17) | ultimo_ano=2001 | padding_ok=True
     OK
[C2] Sem nodata=255 -> OK
[C3] patch.shape=(5, 7, 7) | patch_len=5 -> OK
[C4] label=1 (esp 1) -> OK
[C5] has_21=0.000 | anos_past=0 | cl_tm1=4 -> OK
RESULTADO: PASSOU

AMOSTRA 03 | idx=4506 | row=33364, col=45289 | T=2013 | label=0
───────────────────────────────────────────────────

In [5]:
r"""
stability_check.py
==================
Verifica a estabilidade do MLP embedding rodando 3 vezes
com seeds diferentes.

PRE-REQUISITO — rodar ANTES deste script:
  lulc_dataset.py  (define LULCTransitionDataset)

ORDEM DE EXECUCAO NO NOTEBOOK:
  Celula 1: lulc_dataset.py
  Celula 2: stability_check.py
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score
import json
import rasterio
from rasterio.windows import Window

# LULCTransitionDataset ja esta na memoria (definida em lulc_dataset.py)

INDEX_DIR      = r"D:\Projetos\Cerrado\GeoFM_sampling"
RASTER_CLASSES = r"D:\Projetos\Cerrado\LULC\soy2024_origin_classes_N4_P15_S39_k4_mask.tif"
SC_DATA_DIR    = r"D:\Projetos\Cerrado\LULC"
SC_PATTERN     = "brazil_coverage_{ano}_Cerrado.tif"
SC_YEARS       = list(range(1985, 2025))

SEP = "=" * 70

# ============================================================
# CONFIGURACAO
# ============================================================
SEEDS      = [42, 123, 7]
N_TREINO   = 10000
N_VAL      = 1000
N_TESTE    = 500

BATCH_SIZE = 32
LR         = 1e-3
EPOCHS     = 30
EMBED_DIM  = 8
HIDDEN_DIMS = [128, 64, 32]
DROPOUT    = 0.3
VOCAB_SIZE = 128
SERIE_LEN  = 39

CLASSE_POS = 1
CLASSE_NEG = 2
BLOCK      = 2048
FRAC_TR    = 0.70
FRAC_VA    = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# MODELO
# ============================================================

class MLPEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        input_dim = SERIE_LEN * EMBED_DIM + 3
        layers, prev = [], input_dim
        for h in HIDDEN_DIMS:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(DROPOUT)]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, serie, features):
        emb      = self.embedding(serie.long())
        emb_flat = emb.view(emb.size(0), -1)
        return self.net(torch.cat([emb_flat, features], dim=1))

# ============================================================
# FUNCOES INTERNAS DO STABILITY CHECK
# — nomes com prefixo SC_ para nao conflitar com lulc_dataset.py
# ============================================================

def SC_ler_pixel(year, row, col):
    path = os.path.join(SC_DATA_DIR, SC_PATTERN.format(ano=year))
    with rasterio.open(path) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def SC_encontrar_T(row, col):
    prev = None
    for y in SC_YEARS:
        v = SC_ler_pixel(y, row, col)
        if v in (0, 21):
            prev = None
            continue
        if prev == 4 and v == 15:
            return y
        prev = v
    return None

def SC_coletar_coords():
    with rasterio.open(RASTER_CLASSES) as ds:
        H, W = ds.height, ds.width

    row_tr_end = int(H * FRAC_TR)
    row_va_end = int(H * (FRAC_TR + FRAC_VA))

    def bloco(row):
        if row < row_tr_end: return "treino"
        if row < row_va_end: return "val"
        return "teste"

    coords = {
        "treino": {CLASSE_POS: [], CLASSE_NEG: []},
        "val"   : {CLASSE_POS: [], CLASSE_NEG: []},
        "teste" : {CLASSE_POS: [], CLASSE_NEG: []},
    }

    print("  Varrendo raster...")
    with rasterio.open(RASTER_CLASSES) as ds:
        for row0 in range(0, H, BLOCK):
            bh = min(BLOCK, H - row0)
            for col0 in range(0, W, BLOCK):
                bw = min(BLOCK, W - col0)
                arr = ds.read(1, window=Window(col0, row0, bw, bh))
                for cls in (CLASSE_POS, CLASSE_NEG):
                    rs, cs = np.where(arr == cls)
                    for r, c in zip(rs, cs):
                        coords[bloco(row0 + r)][cls].append((row0 + r, col0 + c))

    print(f"  treino pos={len(coords['treino'][CLASSE_POS]):,} "
          f"neg={len(coords['treino'][CLASSE_NEG]):,} | "
          f"val pos={len(coords['val'][CLASSE_POS]):,} | "
          f"teste pos={len(coords['teste'][CLASSE_POS]):,}")
    return coords

def SC_amostrar(coords, n_total, seed):
    rng  = np.random.default_rng(seed)
    dfs  = {}
    for bloco, n in [("treino", n_total), ("val", N_VAL), ("teste", N_TESTE)]:
        n_cada    = n // 2
        registros = []
        for cls, label in [(CLASSE_POS, 1), (CLASSE_NEG, 0)]:
            lista = coords[bloco][cls]
            idx   = rng.permutation(len(lista))[:min(len(lista), n_cada * 10)]
            for i in idx:
                if len([r for r in registros if r["label"] == label]) >= n_cada:
                    break
                row, col = lista[i]
                T = SC_encontrar_T(row, col)
                if T is None:
                    continue
                registros.append({"row": row, "col": col, "T": T, "label": label})
        dfs[bloco] = pd.DataFrame(registros).sample(frac=1, random_state=seed).reset_index(drop=True)
    return dfs["treino"], dfs["val"], dfs["teste"]

def SC_treinar(df_tr, df_va, df_te, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    ld_tr = DataLoader(LULCTransitionDataset(df_tr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    ld_va = DataLoader(LULCTransitionDataset(df_va), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    ld_te = DataLoader(LULCTransitionDataset(df_te), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model     = MLPEmbedding().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

    melhor_val, melhor_state = 0.0, None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in ld_tr:
            optimizer.zero_grad()
            loss = criterion(
                model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE)),
                batch["label"].to(DEVICE)
            )
            loss.backward()
            optimizer.step()

        model.eval()
        preds_va, labels_va = [], []
        with torch.no_grad():
            for batch in ld_va:
                logits = model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE))
                preds_va.extend(logits.argmax(1).cpu().numpy())
                labels_va.extend(batch["label"].numpy())
        val_acc = balanced_accuracy_score(labels_va, preds_va)
        scheduler.step(val_acc)
        if val_acc > melhor_val:
            melhor_val   = val_acc
            melhor_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(melhor_state)
    model.eval()
    preds_te, labels_te = [], []
    with torch.no_grad():
        for batch in ld_te:
            logits = model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE))
            preds_te.extend(logits.argmax(1).cpu().numpy())
            labels_te.extend(batch["label"].numpy())

    return {
        "val_bal_acc" : round(melhor_val, 4),
        "te_bal_acc"  : round(balanced_accuracy_score(labels_te, preds_te), 4),
        "te_f1"       : round(f1_score(labels_te, preds_te, zero_division=0), 4),
        "n_treino"    : len(df_tr),
        "n_teste"     : len(df_te),
        "n_pos_teste" : int(pd.Series(labels_te).sum()),
    }

# ============================================================
# MAIN
# ============================================================
print(f"\n{SEP}")
print("STABILITY CHECK — MLP EMBEDDING (3 SEEDS)")
print(SEP)
print(f"Device: {DEVICE} | Seeds: {SEEDS} | Epochs: {EPOCHS} | N treino: {N_TREINO}")

print(f"\n[1/2] Coletando coordenadas do raster...")
coords = SC_coletar_coords()

print(f"\n[2/2] Rodando {len(SEEDS)} experimentos...")
resultados = []

for i, seed in enumerate(SEEDS):
    print(f"\n{SEP}")
    print(f"RUN {i+1}/{len(SEEDS)} — seed={seed}")
    print(SEP)

    df_tr, df_va, df_te = SC_amostrar(coords, N_TREINO, seed)
    print(f"  Treino={len(df_tr)} | Val={len(df_va)} | Teste={len(df_te)} "
          f"(pos={df_te['label'].sum()}, {100*df_te['label'].mean():.1f}%)")

    print(f"  Treinando ({EPOCHS} epochs)...")
    res = SC_treinar(df_tr, df_va, df_te, seed)
    res["seed"] = seed
    resultados.append(res)
    print(f"  Val={res['val_bal_acc']:.4f} | Teste={res['te_bal_acc']:.4f} | F1={res['te_f1']:.4f}")

# resumo
print(f"\n{SEP}")
print("RESUMO DE ESTABILIDADE")
print(SEP)

df_res  = pd.DataFrame(resultados)
te_accs = df_res["te_bal_acc"].values
te_f1s  = df_res["te_f1"].values

print(f"\n  {'Seed':>6}  {'N treino':>9}  {'N teste':>8}  {'Pos teste':>10}  "
      f"{'Val':>8}  {'Teste':>8}  {'F1':>8}")
print(f"  {'─'*70}")
for _, r in df_res.iterrows():
    print(f"  {int(r.seed):>6}  {int(r.n_treino):>9}  {int(r.n_teste):>8}  "
          f"{int(r.n_pos_teste):>10}  {r.val_bal_acc:>8.4f}  "
          f"{r.te_bal_acc:>8.4f}  {r.te_f1:>8.4f}")

std = np.std(te_accs)
print(f"\n  Media teste  : {np.mean(te_accs):.4f}")
print(f"  Desvio padrao: {std:.4f}")
print(f"  Min / Max    : {np.min(te_accs):.4f} / {np.max(te_accs):.4f}")
print(f"  Media F1     : {np.mean(te_f1s):.4f} (std={np.std(te_f1s):.4f})")

print(f"\n{SEP}")
if std < 0.03:
    print(f"RESULTADO ESTAVEL (std={std:.4f} < 0.03)")
    print("  Resultado confirmado. Proximo passo: documentar e planejar escala.")
else:
    print(f"RESULTADO FRAGIL (std={std:.4f} >= 0.03)")
    print("  Investigar distribuicao do conjunto de teste.")

out_path = os.path.join(INDEX_DIR, "stability_results.json")
with open(out_path, "w") as f:
    json.dump({
        "seeds": SEEDS,
        "runs" : resultados,
        "summary": {
            "mean_te_bal_acc": round(float(np.mean(te_accs)), 4),
            "std_te_bal_acc" : round(float(np.std(te_accs)),  4),
            "min_te_bal_acc" : round(float(np.min(te_accs)),  4),
            "max_te_bal_acc" : round(float(np.max(te_accs)),  4),
            "mean_te_f1"     : round(float(np.mean(te_f1s)),  4),
            "std_te_f1"      : round(float(np.std(te_f1s)),   4),
            "stable"         : bool(std < 0.03),
        }
    }, f, indent=2)
print(f"Resultados salvos em: {out_path}")
print(SEP)


STABILITY CHECK — MLP EMBEDDING (3 SEEDS)
Device: cuda | Seeds: [42, 123, 7] | Epochs: 30 | N treino: 10000

[1/2] Coletando coordenadas do raster...
  Varrendo raster...
  treino pos=527,553 neg=360,854 | val pos=4,805 | teste pos=170

[2/2] Rodando 3 experimentos...

RUN 1/3 — seed=42
  Treino=10000 | Val=1000 | Teste=420 (pos=170, 40.5%)
  Treinando (30 epochs)...
  Val=0.8190 | Teste=0.5693 | F1=0.2802

RUN 2/3 — seed=123
  Treino=10000 | Val=1000 | Teste=420 (pos=170, 40.5%)
  Treinando (30 epochs)...
  Val=0.7930 | Teste=0.5575 | F1=0.2463

RUN 3/3 — seed=7
  Treino=10000 | Val=1000 | Teste=420 (pos=170, 40.5%)
  Treinando (30 epochs)...
  Val=0.8270 | Teste=0.5329 | F1=0.1818

RESUMO DE ESTABILIDADE

    Seed   N treino   N teste   Pos teste       Val     Teste        F1
  ──────────────────────────────────────────────────────────────────────
      42      10000       420         170    0.8190    0.5693    0.2802
     123      10000       420         170    0.7930    0.5575    

In [9]:
"""
GERADOR DE RASTER DE ÍNDICES ESPACIAIS
=======================================

Cria o arquivo split_indices.tif a partir dos CSVs:
- indice_treino.csv
- indice_val.csv
- indice_teste.csv

Valores no raster:
  0 = sem dados
  1 = treino
  2 = validação
  3 = teste
"""

import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path


def create_split_indices_raster(base_dir, reference_raster_path, output_path):
    """
    Cria raster de índices espaciais a partir dos CSVs.
    
    Args:
        base_dir: Diretório com os CSVs de índices
        reference_raster_path: Raster de referência para dimensões/transform
        output_path: Onde salvar split_indices.tif
    """
    
    print("="*80)
    print("GERANDO RASTER DE ÍNDICES ESPACIAIS")
    print("="*80)
    
    base_dir = Path(base_dir)
    
    # Arquivos de índices
    csv_files = {
        'treino': base_dir / 'indice_treino.csv',
        'val': base_dir / 'indice_val.csv',
        'teste': base_dir / 'indice_teste.csv'
    }
    
    # Verificar arquivos
    print("\n[1/5] Verificando arquivos CSV...")
    for name, path in csv_files.items():
        if path.exists():
            print(f"  ✓ {name}: {path.name}")
        else:
            print(f"  ✗ FALTANDO: {path}")
            return False
    
    # Carregar raster de referência
    print("\n[2/5] Carregando raster de referência...")
    with rasterio.open(reference_raster_path) as src:
        height = src.height
        width = src.width
        transform = src.transform
        crs = src.crs
        
        print(f"  Dimensões: {width} x {height}")
        print(f"  CRS: {crs}")
    
    # Criar array de índices
    print("\n[3/5] Criando array de índices...")
    indices = np.zeros((height, width), dtype='uint8')
    
    # Carregar e processar cada CSV
    print("\n[4/5] Processando CSVs...")
    
    for split_name, csv_path in csv_files.items():
        print(f"\n  Processando {split_name}...")
        
        # Carregar CSV
        df = pd.read_csv(csv_path)
        
        print(f"    Total de linhas: {len(df):,}")
        
        # Determinar valor para este split
        split_value = {'treino': 1, 'val': 2, 'teste': 3}[split_name]
        
        # Verificar colunas
        if 'row' in df.columns and 'col' in df.columns:
            # Formato direto com row/col
            rows = df['row'].values
            cols = df['col'].values
            
        elif 'y' in df.columns and 'x' in df.columns:
            # Formato com coordenadas y/x (assumindo que são índices)
            rows = df['y'].values
            cols = df['x'].values
            
        else:
            print(f"    ⚠ Aviso: Colunas não reconhecidas em {csv_path.name}")
            print(f"       Colunas disponíveis: {df.columns.tolist()}")
            
            # Tentar usar as duas primeiras colunas numéricas
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) >= 2:
                rows = df[numeric_cols[0]].values
                cols = df[numeric_cols[1]].values
                print(f"       Usando: {numeric_cols[0]} e {numeric_cols[1]}")
            else:
                print(f"    ✗ Erro: Não foi possível identificar colunas de coordenadas")
                continue
        
        # Filtrar índices válidos
        valid_mask = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
        rows_valid = rows[valid_mask]
        cols_valid = cols[valid_mask]
        
        if np.sum(~valid_mask) > 0:
            print(f"    ⚠ {np.sum(~valid_mask):,} índices fora dos limites (ignorados)")
        
        print(f"    Aplicando {len(rows_valid):,} índices válidos...")
        
        # Aplicar no array
        indices[rows_valid, cols_valid] = split_value
        
        print(f"    ✓ Concluído")
    
    # Estatísticas
    print("\n  Estatísticas do raster de índices:")
    print(f"    Pixels com treino (1): {np.sum(indices == 1):,}")
    print(f"    Pixels com val (2):    {np.sum(indices == 2):,}")
    print(f"    Pixels com teste (3):  {np.sum(indices == 3):,}")
    print(f"    Pixels sem dados (0):  {np.sum(indices == 0):,}")
    
    # Salvar raster
    print("\n[5/5] Salvando raster...")
    
    profile = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'width': width,
        'height': height,
        'count': 1,
        'crs': crs,
        'transform': transform,
        'nodata': 0,
        'compress': 'lzw'
    }
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(indices, 1)
    
    file_size_mb = Path(output_path).stat().st_size / (1024**2)
    
    print(f"\n✓ Raster salvo: {output_path}")
    print(f"  Tamanho: {file_size_mb:.2f} MB")
    
    print("\n" + "="*80)
    print("RASTER DE ÍNDICES CRIADO COM SUCESSO!")
    print("="*80)
    
    return True


def quick_visualization(indices_path, output_dir):
    """Cria visualização rápida do raster de índices."""
    
    import matplotlib.pyplot as plt
    
    print("\nGerando visualização rápida...")
    
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    
    with rasterio.open(indices_path) as src:
        indices = src.read(1)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Criar colormap customizado
    from matplotlib.colors import ListedColormap
    colors = ['white', 'blue', 'green', 'red']  # 0, 1, 2, 3
    cmap = ListedColormap(colors)
    
    im = ax.imshow(indices, cmap=cmap, vmin=0, vmax=3, aspect='auto')
    ax.set_title('Distribuição Espacial dos Splits', fontsize=14, fontweight='bold')
    ax.set_xlabel('X (longitude)')
    ax.set_ylabel('Y (latitude)')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
    cbar.ax.set_yticklabels(['Sem dados', 'Treino', 'Validação', 'Teste'])
    
    plt.tight_layout()
    plt.savefig(output_dir / 'split_indices_preview.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Preview salvo: {output_dir / 'split_indices_preview.png'}")


# =============================================================================
# EXECUÇÃO
# =============================================================================

if __name__ == "__main__":
    # Configuração
    base_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
    
    # Raster de referência (use qualquer raster LULC para obter dimensões/transform)
    # Baseado na sua imagem, você tem cerrado_mask_aligned_to_LULC.tif
    reference_raster = base_dir / "cerrado_mask_aligned_to_LULC.tif"
    
    # Se não tiver esse, pode usar outro arquivo .tif de referência
    # Descomente a linha abaixo e ajuste:
    # reference_raster = Path(r"D:\Projetos\LULC\algum_raster.tif")
    
    output_path = base_dir / "split_indices.tif"
    
    print("\n" + "="*80)
    print("CONFIGURAÇÃO")
    print("="*80)
    print(f"\nDiretório base: {base_dir}")
    print(f"Raster referência: {reference_raster.name}")
    print(f"Saída: {output_path.name}")
    
    # Verificar raster de referência
    if not reference_raster.exists():
        print(f"\n✗ ERRO: Raster de referência não encontrado:")
        print(f"  {reference_raster}")
        print("\nOPÇÕES:")
        print("  1. Ajuste o caminho 'reference_raster' no script")
        print("  2. Use qualquer arquivo .tif da área de estudo")
        print("     (precisa ter mesmas dimensões que os dados de treino)")
    else:
        # Criar raster
        success = create_split_indices_raster(base_dir, reference_raster, output_path)
        
        if success:
            # Criar visualização
            quick_visualization(output_path, base_dir / "diagnostics")
            
            print("\n" + "="*80)
            print("PRÓXIMO PASSO:")
            print("="*80)
            print("\nAgora você pode executar o script de diagnóstico:")
            print("  python diagnostic_spatial_overfitting.py")
            print("\n" + "="*80)


CONFIGURAÇÃO

Diretório base: D:\Projetos\Cerrado\GeoFM_sampling
Raster referência: cerrado_mask_aligned_to_LULC.tif
Saída: split_indices.tif
GERANDO RASTER DE ÍNDICES ESPACIAIS

[1/5] Verificando arquivos CSV...
  ✓ treino: indice_treino.csv
  ✓ val: indice_val.csv
  ✓ teste: indice_teste.csv

[2/5] Carregando raster de referência...
  Dimensões: 71227 x 82933
  CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]

[3/5] Criando array de índices...

[4/5] Processando CSVs...

  Processando treino...
    Total de linhas: 10,000
    Aplicando 10,000 índices válidos...
    ✓ Concluído

  Processando val...
    Total de linhas: 1,000
    Aplicando 1,000 índices válidos...
    ✓ Concluído

  Processando teste...
    Total de linhas: 420
    Aplicando 420 índices válidos...
    ✓ Concluído

  Estatísticas do raster de í

In [10]:
"""
ANÁLISE DE DISTRIBUIÇÃO ESPACIAL E DIAGNÓSTICO
==============================================

Analisa a distribuição geográfica dos conjuntos treino/val/teste
e identifica possíveis causas do gap de performance.

Autor: Diagnóstico de Overfitting Espacial
"""

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns


def load_spatial_indices(indices_path):
    """Carrega índices espaciais do arquivo gerado pelo splitter."""
    with rasterio.open(indices_path) as src:
        indices = src.read(1)
    return indices


def analyze_spatial_distribution(indices_path, output_dir):
    """
    Analisa distribuição espacial dos splits.
    
    Args:
        indices_path: Caminho para o raster de índices (split_indices.tif)
        output_dir: Diretório para salvar análises
    """
    
    print("="*80)
    print("ANÁLISE DE DISTRIBUIÇÃO ESPACIAL")
    print("="*80)
    
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # Carregar índices
    print("\n[1/5] Carregando índices espaciais...")
    indices = load_spatial_indices(indices_path)
    
    # Criar máscaras
    mask_treino = (indices == 1)
    mask_val = (indices == 2)
    mask_teste = (indices == 3)
    
    # Estatísticas básicas
    print("\n[2/5] Estatísticas de distribuição:")
    print(f"  Treino: {mask_treino.sum():,} pixels")
    print(f"  Val:    {mask_val.sum():,} pixels")
    print(f"  Teste:  {mask_teste.sum():,} pixels")
    print(f"  Total:  {(mask_treino | mask_val | mask_teste).sum():,} pixels")
    
    # Análise de cobertura espacial
    print("\n[3/5] Analisando cobertura espacial...")
    
    height, width = indices.shape
    
    # Dividir em quadrantes (4x4 grid)
    n_rows, n_cols = 4, 4
    quad_h = height // n_rows
    quad_w = width // n_cols
    
    quadrant_stats = []
    
    for i in range(n_rows):
        for j in range(n_cols):
            # Definir quadrante
            r_start = i * quad_h
            r_end = (i + 1) * quad_h if i < n_rows - 1 else height
            c_start = j * quad_w
            c_end = (j + 1) * quad_w if j < n_cols - 1 else width
            
            # Extrair quadrante
            quad_treino = mask_treino[r_start:r_end, c_start:c_end].sum()
            quad_val = mask_val[r_start:r_end, c_start:c_end].sum()
            quad_teste = mask_teste[r_start:r_end, c_start:c_end].sum()
            quad_total = quad_treino + quad_val + quad_teste
            
            if quad_total > 0:
                quadrant_stats.append({
                    'quadrant': f"{i},{j}",
                    'treino_pct': 100 * quad_treino / quad_total,
                    'val_pct': 100 * quad_val / quad_total,
                    'teste_pct': 100 * quad_teste / quad_total,
                    'total': quad_total
                })
    
    # Imprimir estatísticas de quadrantes
    print("\n  Distribuição por quadrante (4x4):")
    print("  Quadrante | Treino% | Val% | Teste% | Total pixels")
    print("  " + "-"*56)
    
    for stat in quadrant_stats[:10]:  # Mostrar primeiros 10
        print(f"  {stat['quadrant']:>9} | {stat['treino_pct']:>6.1f}% | "
              f"{stat['val_pct']:>3.1f}% | {stat['teste_pct']:>5.1f}% | "
              f"{stat['total']:>12,}")
    
    if len(quadrant_stats) > 10:
        print(f"  ... e mais {len(quadrant_stats)-10} quadrantes")
    
    # Calcular concentração espacial
    print("\n[4/5] Calculando concentração espacial...")
    
    # Centro de massa de cada conjunto
    def center_of_mass(mask):
        y_coords, x_coords = np.where(mask)
        if len(y_coords) == 0:
            return None, None
        return np.mean(y_coords), np.mean(x_coords)
    
    treino_cy, treino_cx = center_of_mass(mask_treino)
    val_cy, val_cx = center_of_mass(mask_val)
    teste_cy, teste_cx = center_of_mass(mask_teste)
    
    print(f"\n  Centros de massa (y, x):")
    print(f"    Treino: ({treino_cy:.1f}, {treino_cx:.1f})")
    print(f"    Val:    ({val_cy:.1f}, {val_cx:.1f})")
    print(f"    Teste:  ({teste_cy:.1f}, {teste_cx:.1f})")
    
    # Distância entre centros
    if None not in [treino_cy, val_cy, teste_cy]:
        dist_treino_val = np.sqrt((treino_cy - val_cy)**2 + (treino_cx - val_cx)**2)
        dist_treino_teste = np.sqrt((treino_cy - teste_cy)**2 + (treino_cx - teste_cx)**2)
        dist_val_teste = np.sqrt((val_cy - teste_cy)**2 + (val_cx - teste_cx)**2)
        
        print(f"\n  Distâncias entre centros (pixels):")
        print(f"    Treino ↔ Val:   {dist_treino_val:,.1f}")
        print(f"    Treino ↔ Teste: {dist_treino_teste:,.1f}")
        print(f"    Val ↔ Teste:    {dist_val_teste:,.1f}")
        
        # Diagnóstico
        if dist_treino_teste > 2 * dist_treino_val:
            print("\n  ⚠️ ALERTA: Teste está significativamente mais distante que Val!")
            print("     Isso pode explicar o gap de performance.")
    
    # Visualizações
    print("\n[5/5] Gerando visualizações...")
    
    # Plot 1: Distribuição espacial
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Subplot 1: Treino
    y_treino, x_treino = np.where(mask_treino)
    if len(y_treino) > 0:
        axes[0, 0].scatter(x_treino[::10], y_treino[::10], alpha=0.3, s=1, c='blue')
        axes[0, 0].set_title(f'Treino (n={mask_treino.sum():,})', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('X (longitude)')
        axes[0, 0].set_ylabel('Y (latitude)')
        axes[0, 0].invert_yaxis()
    
    # Subplot 2: Validação
    y_val, x_val = np.where(mask_val)
    if len(y_val) > 0:
        axes[0, 1].scatter(x_val[::10], y_val[::10], alpha=0.3, s=1, c='green')
        axes[0, 1].set_title(f'Validação (n={mask_val.sum():,})', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('X (longitude)')
        axes[0, 1].set_ylabel('Y (latitude)')
        axes[0, 1].invert_yaxis()
    
    # Subplot 3: Teste
    y_teste, x_teste = np.where(mask_teste)
    if len(y_teste) > 0:
        axes[1, 0].scatter(x_teste[::10], y_teste[::10], alpha=0.3, s=1, c='red')
        axes[1, 0].set_title(f'Teste (n={mask_teste.sum():,})', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('X (longitude)')
        axes[1, 0].set_ylabel('Y (latitude)')
        axes[1, 0].invert_yaxis()
    
    # Subplot 4: Todos juntos
    if len(y_treino) > 0:
        axes[1, 1].scatter(x_treino[::50], y_treino[::50], alpha=0.2, s=1, c='blue', label='Treino')
    if len(y_val) > 0:
        axes[1, 1].scatter(x_val[::50], y_val[::50], alpha=0.2, s=1, c='green', label='Val')
    if len(y_teste) > 0:
        axes[1, 1].scatter(x_teste[::50], y_teste[::50], alpha=0.2, s=1, c='red', label='Teste')
    
    axes[1, 1].set_title('Todos os Conjuntos', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('X (longitude)')
    axes[1, 1].set_ylabel('Y (latitude)')
    axes[1, 1].legend()
    axes[1, 1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(output_dir / 'distribuicao_espacial_splits.png', dpi=150, bbox_inches='tight')
    print(f"  ✓ Salvo: {output_dir / 'distribuicao_espacial_splits.png'}")
    plt.close()
    
    # Plot 2: Mapa de calor de densidade
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, (name, mask, color) in enumerate([
        ('Treino', mask_treino, 'Blues'),
        ('Validação', mask_val, 'Greens'),
        ('Teste', mask_teste, 'Reds')
    ]):
        # Criar heatmap de densidade (reduzir resolução)
        scale = 10
        h_small = height // scale
        w_small = width // scale
        
        density = np.zeros((h_small, w_small))
        for i in range(h_small):
            for j in range(w_small):
                density[i, j] = mask[i*scale:(i+1)*scale, j*scale:(j+1)*scale].sum()
        
        im = axes[idx].imshow(density, cmap=color, aspect='auto')
        axes[idx].set_title(f'{name}\n(densidade por {scale}x{scale} pixels)', 
                           fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('X')
        axes[idx].set_ylabel('Y')
        plt.colorbar(im, ax=axes[idx], label='Densidade')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'densidade_espacial_splits.png', dpi=150, bbox_inches='tight')
    print(f"  ✓ Salvo: {output_dir / 'densidade_espacial_splits.png'}")
    plt.close()
    
    # Salvar estatísticas em JSON
    stats_output = {
        'total_pixels': {
            'treino': int(mask_treino.sum()),
            'val': int(mask_val.sum()),
            'teste': int(mask_teste.sum())
        },
        'center_of_mass': {
            'treino': {'y': float(treino_cy), 'x': float(treino_cx)},
            'val': {'y': float(val_cy), 'x': float(val_cx)},
            'teste': {'y': float(teste_cy), 'x': float(teste_cx)}
        },
        'distances': {
            'treino_val': float(dist_treino_val),
            'treino_teste': float(dist_treino_teste),
            'val_teste': float(dist_val_teste)
        },
        'quadrant_stats': quadrant_stats
    }
    
    with open(output_dir / 'spatial_analysis.json', 'w') as f:
        json.dump(stats_output, f, indent=2)
    
    print(f"  ✓ Salvo: {output_dir / 'spatial_analysis.json'}")
    
    print("\n" + "="*80)
    print("ANÁLISE CONCLUÍDA")
    print("="*80)
    
    return stats_output


def analyze_model_predictions(stability_results_path, output_dir):
    """
    Analisa predições do modelo a partir dos resultados de estabilidade.
    
    Args:
        stability_results_path: Caminho para stability_results.json
        output_dir: Diretório para salvar análises
    """
    
    print("\n" + "="*80)
    print("ANÁLISE DE PREDIÇÕES DO MODELO")
    print("="*80)
    
    output_dir = Path(output_dir)
    
    # Carregar resultados
    with open(stability_results_path, 'r') as f:
        results = json.load(f)
    
    print("\n[1/2] Resumo de Performance:")
    print("\n  Seed | Val Acc | Test Acc | F1 Score | Gap (Val-Test)")
    print("  " + "-"*60)
    
    for run in results['runs']:
        gap = run['val_acc'] - run['test_acc']
        print(f"  {run['seed']:>4} | {run['val_acc']:>7.4f} | "
              f"{run['test_acc']:>8.4f} | {run['f1_score']:>8.4f} | "
              f"{gap:>14.4f} ({gap*100:>5.1f}pp)")
    
    # Calcular médias
    avg_gap = np.mean([r['val_acc'] - r['test_acc'] for r in results['runs']])
    
    print(f"\n  MÉDIA DE GAP: {avg_gap:.4f} ({avg_gap*100:.1f} pontos percentuais)")
    
    if avg_gap > 0.20:
        print("\n  🚨 ALERTA CRÍTICO: Gap > 20pp indica overfitting severo!")
        print("     Possíveis causas:")
        print("     1. Distribuição espacial muito diferente entre val e teste")
        print("     2. Padrões temporais regionais não capturados")
        print("     3. Overfitting aos padrões específicos da região de validação")
    
    # Visualização
    print("\n[2/2] Gerando visualizações...")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Comparação Val vs Test
    seeds = [r['seed'] for r in results['runs']]
    val_accs = [r['val_acc'] for r in results['runs']]
    test_accs = [r['test_acc'] for r in results['runs']]
    
    x = np.arange(len(seeds))
    width = 0.35
    
    axes[0].bar(x - width/2, val_accs, width, label='Validação', alpha=0.8)
    axes[0].bar(x + width/2, test_accs, width, label='Teste', alpha=0.8)
    axes[0].set_xlabel('Seed')
    axes[0].set_ylabel('Acurácia')
    axes[0].set_title('Validação vs Teste por Seed', fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(seeds)
    axes[0].legend()
    axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random (50%)')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Gap por seed
    gaps = [v - t for v, t in zip(val_accs, test_accs)]
    
    axes[1].bar(x, gaps, alpha=0.8, color='red')
    axes[1].set_xlabel('Seed')
    axes[1].set_ylabel('Gap (Val - Test)')
    axes[1].set_title('Overfitting Gap por Seed', fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(seeds)
    axes[1].axhline(y=0.15, color='orange', linestyle='--', alpha=0.5, 
                    label='Limite Aceitável (15pp)')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'performance_analysis.png', dpi=150, bbox_inches='tight')
    print(f"  ✓ Salvo: {output_dir / 'performance_analysis.png'}")
    plt.close()
    
    print("\n" + "="*80)


# =============================================================================
# EXECUÇÃO
# =============================================================================

if __name__ == "__main__":
    # Caminhos - AJUSTE CONFORME NECESSÁRIO
    BASE_DIR = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
    
    indices_path = BASE_DIR / "split_indices.tif"
    stability_results_path = BASE_DIR / "stability_results.json"
    output_dir = BASE_DIR / "diagnostics"
    
    print("\n" + "="*80)
    print("DIAGNÓSTICO COMPLETO DE OVERFITTING ESPACIAL")
    print("="*80)
    
    # Verificar arquivos
    print("\nVerificando arquivos necessários:")
    
    if indices_path.exists():
        print(f"  ✓ Índices espaciais: {indices_path.name}")
    else:
        print(f"  ✗ FALTANDO: {indices_path}")
        print("\n  Execute primeiro o script de split espacial!")
        exit(1)
    
    if stability_results_path.exists():
        print(f"  ✓ Resultados de estabilidade: {stability_results_path.name}")
    else:
        print(f"  ✗ FALTANDO: {stability_results_path}")
        print("\n  Execute primeiro o teste de estabilidade!")
        exit(1)
    
    # Executar análises
    print("\n" + "="*80)
    
    # 1. Análise espacial
    spatial_stats = analyze_spatial_distribution(indices_path, output_dir)
    
    # 2. Análise de predições
    analyze_model_predictions(stability_results_path, output_dir)
    
    print("\n" + "="*80)
    print("✅ DIAGNÓSTICO COMPLETO!")
    print("="*80)
    print(f"\nResultados salvos em: {output_dir}")
    print("\nArquivos gerados:")
    print("  • distribuicao_espacial_splits.png")
    print("  • densidade_espacial_splits.png")
    print("  • performance_analysis.png")
    print("  • spatial_analysis.json")
    print("\n" + "="*80)


DIAGNÓSTICO COMPLETO DE OVERFITTING ESPACIAL

Verificando arquivos necessários:
  ✓ Índices espaciais: split_indices.tif
  ✓ Resultados de estabilidade: stability_results.json

ANÁLISE DE DISTRIBUIÇÃO ESPACIAL

[1/5] Carregando índices espaciais...

[2/5] Estatísticas de distribuição:
  Treino: 10,000 pixels
  Val:    1,000 pixels
  Teste:  420 pixels
  Total:  11,420 pixels

[3/5] Analisando cobertura espacial...

  Distribuição por quadrante (4x4):
  Quadrante | Treino% | Val% | Teste% | Total pixels
  --------------------------------------------------------
        0,2 |  100.0% | 0.0% |   0.0% |          292
        0,3 |  100.0% | 0.0% |   0.0% |          597
        1,0 |  100.0% | 0.0% |   0.0% |          408
        1,1 |  100.0% | 0.0% |   0.0% |          275
        1,2 |  100.0% | 0.0% |   0.0% |        5,985
        1,3 |  100.0% | 0.0% |   0.0% |          118
        2,0 |  100.0% | 0.0% |   0.0% |          893
        2,1 |   80.3% | 19.7% |   0.0% |          956
       

TypeError: Object of type int32 is not JSON serializable

In [15]:
"""
EXECUTAR ESTE SCRIPT NO SEU JUPYTER
====================================

Cole este código numa célula e execute para congelar o baseline atual.

IMPORTANTE: Ajuste as variáveis no início do script conforme seu código!
"""

import torch
import numpy as np
from pathlib import Path
from freeze_baseline import freeze_current_baseline

# =============================================================================
# CONFIGURAÇÃO - AJUSTE ESTAS VARIÁVEIS
# =============================================================================

print("="*80)
print("CONGELANDO BASELINE - MLP EMBEDDING")
print("="*80)

# 1. CAMINHO DO MODELO TREINADO
# Se você salvou o modelo durante o treino:
modelo_path = Path(r"D:\Projetos\Cerrado\GeoFM_sampling\models\mlp_embedding_historico.pth")

# Ou se o modelo está em memória (variável 'modelo'):
# modelo = modelo  # já está carregado

# 2. CAMINHO PARA SALVAR O BASELINE
baseline_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling\baseline")

print(f"\n📁 Baseline será salvo em: {baseline_dir}")

# =============================================================================
# CONFIGURAÇÕES DO MODELO
# =============================================================================

# Baseado no seu código mlp_embedding.py
config = {
    'model': {
        'name': 'MLP_Embedding',
        'architecture': 'MLP com LULC Class Embeddings',
        
        # Dimensões de entrada
        'lulc_sequence_length': 39,  # T-39 até T-1
        'num_lulc_classes': 35,      # Classes MapBiomas
        'embedding_dim': 16,          # Embedding de classes
        
        'spatial_patch_shape': (5, 7, 7),  # 5 canais × 7×7 spatial
        'spatial_features': 245,           # 5 × 7 × 7
        
        'aux_features': 3,  # T, lat, lon (normalizados)
        
        'total_input_dim': 624 + 245 + 3,  # 872
        
        # Arquitetura da rede
        'hidden_layers': [256, 128, 64],
        'dropout': 0.3,
        'activation': 'ReLU',
        'output_activation': 'Sigmoid (via BCEWithLogitsLoss)'
    },
    
    'training': {
        'epochs': 30,
        'batch_size': 256,
        'learning_rate': 0.001,
        'weight_decay': 0.0,  # Adicione se você usou
        'optimizer': 'Adam',
        'loss_function': 'BCEWithLogitsLoss',
        'scheduler': None,  # Ou o que você usou
        'early_stopping': False,
        'clip_grad_norm': None  # Ou o valor se você usou
    },
    
    'data': {
        'dataset_path': r"D:\Projetos\Cerrado\LULC",
        'output_dir': r"D:\Projetos\Cerrado\GeoFM_sampling",
        
        # Tamanhos dos conjuntos
        'n_train': 10000,
        'n_val': 1000,
        'n_test': 420,
        'test_positives': 170,
        
        # Estratégia de split
        'split_method': 'spatial_block',
        'stratified_by_T': True,
        
        # Seeds usados no teste de estabilidade
        'seeds': [42, 123, 7]
    },
    
    'preprocessing': {
        'lulc_embedding': True,  # vs. one-hot ou raw
        'spatial_normalization': 'z-score',  # ou outro
        'aux_normalization': True,
        'T_normalization': 'min-max',
        'lat_lon_normalization': 'min-max'
    }
}

# =============================================================================
# MÉTRICAS DE PERFORMANCE
# =============================================================================

# Baseado no seu teste de estabilidade
metrics = {
    'summary': {
        'val_acc_mean': 0.8130,
        'val_acc_std': 0.0132,
        'test_acc_mean': 0.5532,
        'test_acc_std': 0.0152,
        'f1_mean': 0.2361,
        'f1_std': 0.0408,
        'gap_val_test': 0.2598  # 26pp
    },
    
    'runs': [
        {
            'seed': 42,
            'n_train': 10000,
            'n_val': 1000,
            'n_test': 420,
            'test_positives': 170,
            'val_acc': 0.8190,
            'test_acc': 0.5693,
            'f1_score': 0.2802
        },
        {
            'seed': 123,
            'n_train': 10000,
            'n_val': 1000,
            'n_test': 420,
            'test_positives': 170,
            'val_acc': 0.7930,
            'test_acc': 0.5575,
            'f1_score': 0.2463
        },
        {
            'seed': 7,
            'n_train': 10000,
            'n_val': 1000,
            'n_test': 420,
            'test_positives': 170,
            'val_acc': 0.8270,
            'test_acc': 0.5329,
            'f1_score': 0.1818
        }
    ],
    
    'spatial_analysis': {
        'problem_identified': 'Spatial overfitting',
        'train_center_y': 33399.2,
        'train_center_x': 39722.7,
        'test_center_y': 72651.3,
        'test_center_x': 19155.1,
        'distance_train_test': 44314.3,  # pixels
        'interpretation': 'Model trained on dispersed pattern (North), tested on cluster pattern (South)'
    },
    
    'stability': {
        'reproducible': True,
        'std_across_seeds': 0.0152,
        'verdict': 'STABLE (std < 0.03)'
    }
}

# =============================================================================
# ÍNDICES DE DADOS
# =============================================================================

# Se você tem os índices salvos ou em memória:
print("\n[INFO] Carregando índices de dados...")

try:
    # Opção 1: Se você tem arrays/listas em memória
    # train_indices = indice_treino  # sua variável
    # val_indices = indice_val
    # test_indices = indice_teste
    
    # Opção 2: Se estão em arquivos CSV
    import pandas as pd
    
    csv_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
    
    df_train = pd.read_csv(csv_dir / "indice_treino.csv")
    df_val = pd.read_csv(csv_dir / "indice_val.csv")
    df_test = pd.read_csv(csv_dir / "indice_teste.csv")
    
    # Extrair índices (ajuste conforme estrutura do CSV)
    # Se o CSV tem colunas 'row' e 'col', criar índice único:
    if 'row' in df_train.columns and 'col' in df_train.columns:
        # Criar índice composto (row * 1000000 + col)
        train_indices = (df_train['row'].values * 1000000 + df_train['col'].values).tolist()
        val_indices = (df_val['row'].values * 1000000 + df_val['col'].values).tolist()
        test_indices = (df_test['row'].values * 1000000 + df_test['col'].values).tolist()
    else:
        # Ou se tem índice direto
        train_indices = df_train.index.tolist()
        val_indices = df_val.index.tolist()
        test_indices = df_test.index.tolist()
    
    print(f"  ✓ Treino: {len(train_indices):,} índices")
    print(f"  ✓ Val: {len(val_indices):,} índices")
    print(f"  ✓ Teste: {len(test_indices):,} índices")
    
except Exception as e:
    print(f"  ⚠ Aviso: Não foi possível carregar índices ({e})")
    print("     Continuando sem salvar índices...")
    train_indices = None
    val_indices = None
    test_indices = None

# =============================================================================
# CARREGAR MODELO
# =============================================================================

print("\n[INFO] Carregando modelo...")

# Opção 1: Se o modelo já está em memória (variável 'modelo')
try:
    modelo  # tenta acessar a variável
    print("  ✓ Modelo já está em memória")
    
except NameError:
    # Opção 2: Carregar do arquivo
    print(f"  Carregando de: {modelo_path}")
    
    # AJUSTE esta parte conforme sua classe de modelo!
    # Exemplo (você precisa ter a definição da classe):
    """
    from seu_script import MLPEmbedding  # ou onde está definido
    
    modelo = MLPEmbedding(
        num_classes=35,
        embedding_dim=16,
        spatial_dim=245,
        aux_dim=3,
        hidden_dims=[256, 128, 64],
        dropout=0.3
    )
    
    checkpoint = torch.load(modelo_path)
    modelo.load_state_dict(checkpoint)
    """
    
    # Se não conseguir carregar, pelo menos salvar as configs
    print("  ⚠ IMPORTANTE: Ajuste o código acima para carregar seu modelo!")
    print("     Salvando apenas configurações por enquanto...")
    modelo = None

# =============================================================================
# CONGELAR BASELINE
# =============================================================================

print("\n" + "="*80)
print("EXECUTANDO CONGELAMENTO")
print("="*80)

baseline_path = freeze_current_baseline(
    model=modelo,  # pode ser None temporariamente
    config=config,
    metrics=metrics,
    train_idx=train_indices,
    val_idx=val_indices,
    test_idx=test_indices,
    baseline_dir=baseline_dir
)

print("\n" + "="*80)
print("✅ BASELINE CONGELADO!")
print("="*80)
print(f"\n📁 Localização: {baseline_path}")
print("\n📋 Próximos passos:")
print("  1. ✓ Baseline documentado e salvo")
print("  2. ⏳ Revisar README.md no diretório baseline")
print("  3. ⏳ Executar análise de Moran I (1985-2024)")
print("  4. ⏳ Adicionar features espaciais")
print("  5. ⏳ Comparar nova versão com baseline")
print("\n" + "="*80)

ModuleNotFoundError: No module named 'freeze_baseline'

In [17]:
import pandas as pd

df = pd.read_csv(r"D:\Projetos\Cerrado\GeoFM_sampling\indice_treino.csv")
print("Colunas:", df.columns.tolist())
print("\nPrimeiras linhas:")
print(df.head())

Colunas: ['row', 'col', 'T', 'label']

Primeiras linhas:
     row    col     T  label
0  46555  47852  2013      0
1  30680  45403  2012      1
2  21052  50059  2016      1
3  47839  44254  2014      1
4  37827  59987  2018      1


In [20]:
"""
INTEGRAÇÃO PADRÕES ESPACIAIS → DATASET ML
==========================================

VERSÃO CORRETA - Usando rasters brazil_coverage_*_Cerrado.tif

Input:
  - CSVs com (row, col, T, label)
  - Rasters brazil_coverage_{ano}_Cerrado.tif (para transform)
  - Shapefile hexagonal com padrões espaciais
  
Output:
  - CSVs expandidos com features espaciais

Autor: Integração Correta
Data: 2026-03-07
"""

import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from pathlib import Path
from shapely.geometry import Point
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


def get_transform_from_raster(lulc_dir, year=1985):
    """
    Obtém transform de um raster LULC.
    
    Args:
        lulc_dir: Diretório com rasters
        year: Ano do raster
        
    Returns:
        transform, crs
    """
    raster_path = Path(lulc_dir) / f"brazil_coverage_{year}_Cerrado.tif"
    
    print(f"\nCarregando transform do raster:")
    print(f"  {raster_path}")
    
    if not raster_path.exists():
        print(f"  ✗ Raster não encontrado!")
        print(f"\n  Tentando alternativas...")
        
        # Tentar outros anos
        for alt_year in [1986, 1987, 1988, 1990, 2000]:
            alt_path = Path(lulc_dir) / f"brazil_coverage_{alt_year}_Cerrado.tif"
            if alt_path.exists():
                print(f"  ✓ Usando: {alt_path.name}")
                raster_path = alt_path
                break
        else:
            raise FileNotFoundError(f"Nenhum raster encontrado em {lulc_dir}")
    
    with rasterio.open(raster_path) as src:
        transform = src.transform
        crs = src.crs
        bounds = src.bounds
        
        print(f"  ✓ CRS: {crs}")
        print(f"  ✓ Bounds: {bounds}")
    
    return transform, crs


def rowcol_to_latlon(row, col, transform):
    """
    Converte índices (row, col) para coordenadas (lat, lon).
    
    Args:
        row, col: Índices do pixel
        transform: Affine transform do raster
        
    Returns:
        lat, lon
    """
    # Centro do pixel
    lon, lat = transform * (col + 0.5, row + 0.5)
    return lat, lon


def decode_pattern(code):
    """Decodifica código de padrão."""
    if pd.isna(code) or code == 'NC' or code == '':
        return {
            'has_conversion': 0,
            'is_cluster_conversion': 0,
            'land_use_type': 'none',
            'is_cluster_use': 0,
            'consolidation_level': 0,
            'pattern_code': 'NC'
        }
    
    code = str(code).strip().upper()
    
    # Código "CD" ou "CC" sozinho
    if code in ['CD', 'CC']:
        return {
            'has_conversion': 1,
            'is_cluster_conversion': 1 if code == 'CC' else 0,
            'land_use_type': 'undefined',
            'is_cluster_use': 0,
            'consolidation_level': 2 if code == 'CC' else 1,
            'pattern_code': code
        }
    
    # Código completo (4 letras)
    if len(code) == 4:
        conversion = code[:2]
        land_use = code[2:]
    else:
        return {
            'has_conversion': 0,
            'is_cluster_conversion': 0,
            'land_use_type': 'unknown',
            'is_cluster_use': 0,
            'consolidation_level': 0,
            'pattern_code': code
        }
    
    # Decodificar
    features = {
        'has_conversion': 1,
        'is_cluster_conversion': 1 if conversion == 'CC' else 0,
        'pattern_code': code
    }
    
    if land_use.startswith('A'):
        features['land_use_type'] = 'agriculture'
        features['is_cluster_use'] = 1 if land_use == 'AC' else 0
    elif land_use.startswith('P'):
        features['land_use_type'] = 'pasture'
        features['is_cluster_use'] = 1 if land_use == 'PC' else 0
    else:
        features['land_use_type'] = 'undefined'
        features['is_cluster_use'] = 0
    
    # Consolidação
    if conversion == 'CC' and land_use in ['AC', 'PC']:
        features['consolidation_level'] = 3
    elif conversion == 'CC' or land_use in ['AC', 'PC']:
        features['consolidation_level'] = 2
    else:
        features['consolidation_level'] = 1
    
    return features


def extract_pattern_for_point(lat, lon, gdf_hex, period_col='Class8590', 
                              raster_crs='EPSG:4326'):
    """
    Extrai padrão espacial para um ponto (lat, lon).
    
    Args:
        lat, lon: Coordenadas (no CRS do raster)
        gdf_hex: GeoDataFrame hexagonal
        period_col: Coluna do período
        raster_crs: CRS das coordenadas
        
    Returns:
        Dict com features
    """
    from geopandas import GeoSeries
    
    # Criar ponto
    point_geoseries = GeoSeries([Point(lon, lat)], crs=raster_crs)
    
    # Reprojetar se necessário
    if gdf_hex.crs != raster_crs:
        point_geoseries = point_geoseries.to_crs(gdf_hex.crs)
    
    point = point_geoseries.iloc[0]
    
    # Encontrar hexágono
    contains = gdf_hex.contains(point)
    
    if contains.sum() == 0:
        # Mais próximo
        distances = gdf_hex.distance(point)
        idx = distances.idxmin()
    else:
        idx = gdf_hex[contains].index[0]
    
    # Extrair padrão
    pattern_code = gdf_hex.loc[idx, period_col]
    
    # Decodificar
    features = decode_pattern(pattern_code)
    
    return features


def process_dataset(csv_path, lulc_dir, gdf_hex, output_path, 
                   period_col='Class8590', dataset_name='treino'):
    """
    Processa dataset de ML adicionando padrões espaciais.
    
    Args:
        csv_path: CSV com (row, col, T, label)
        lulc_dir: Diretório com rasters LULC
        gdf_hex: GeoDataFrame hexagonal
        output_path: Onde salvar
        period_col: Coluna do período
        dataset_name: Nome do dataset
        
    Returns:
        DataFrame expandido
    """
    print(f"\n{'='*80}")
    print(f"PROCESSANDO: {dataset_name.upper()}")
    print('='*80)
    
    # Carregar CSV
    print(f"\nCarregando CSV: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"  ✓ Amostras: {len(df):,}")
    print(f"  Colunas: {df.columns.tolist()}")
    
    # Obter transform
    transform, raster_crs = get_transform_from_raster(lulc_dir)
    
    # Converter row/col → lat/lon e extrair padrões
    print(f"\n  Extraindo padrões espaciais...")
    
    features_list = []
    lats = []
    lons = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="  Processando"):
        r = row['row']
        c = row['col']
        
        # Converter para lat/lon
        lat, lon = rowcol_to_latlon(r, c, transform)
        lats.append(lat)
        lons.append(lon)
        
        # Extrair padrão
        pattern_features = extract_pattern_for_point(
            lat, lon, gdf_hex, period_col, raster_crs
        )
        
        features_list.append(pattern_features)
    
    # Adicionar lat/lon
    df['lat'] = lats
    df['lon'] = lons
    
    # Adicionar features espaciais
    df_features = pd.DataFrame(features_list)
    df_expanded = pd.concat([df, df_features], axis=1)
    
    print(f"\n  ✓ Features adicionadas: {len(df_features.columns) + 2}")  # +2 para lat/lon
    print(f"\n  Novas colunas:")
    print(f"    • lat")
    print(f"    • lon")
    for col in df_features.columns:
        print(f"    • {col}")
    
    # Salvar
    df_expanded.to_csv(output_path, index=False)
    
    file_size_mb = Path(output_path).stat().st_size / (1024**2)
    print(f"\n  ✓ Salvo: {output_path.name} ({file_size_mb:.1f} MB)")
    
    return df_expanded


def analyze_patterns(df_train, df_val, df_test):
    """Analisa distribuição de padrões."""
    print("\n" + "="*80)
    print("ANÁLISE DE DISTRIBUIÇÃO DE PADRÕES")
    print("="*80)
    
    print("\n[1] DISTRIBUIÇÃO DE CÓDIGOS:")
    print(f"\n{'Padrão':10s} {'Treino':>15s} {'Val':>15s} {'Teste':>15s}")
    print("-"*60)
    
    all_patterns = set(df_train['pattern_code'].unique()) | \
                   set(df_val['pattern_code'].unique()) | \
                   set(df_test['pattern_code'].unique())
    
    for pattern in sorted(all_patterns):
        count_train = (df_train['pattern_code'] == pattern).sum()
        count_val = (df_val['pattern_code'] == pattern).sum()
        count_test = (df_test['pattern_code'] == pattern).sum()
        
        pct_train = 100 * count_train / len(df_train)
        pct_val = 100 * count_val / len(df_val)
        pct_test = 100 * count_test / len(df_test)
        
        print(f"{pattern:10s} {pct_train:6.1f}% ({count_train:5d}) "
              f"{pct_val:6.1f}% ({count_val:4d}) "
              f"{pct_test:6.1f}% ({count_test:4d})")
    
    print("\n[2] FEATURES AGREGADAS:")
    print(f"\n{'Feature':25s} {'Treino':>10s} {'Val':>10s} {'Teste':>10s}")
    print("-"*60)
    
    for feature in ['is_cluster_conversion', 'is_cluster_use', 'consolidation_level']:
        mean_train = df_train[feature].mean()
        mean_val = df_val[feature].mean()
        mean_test = df_test[feature].mean()
        
        print(f"{feature:25s} {mean_train:10.3f} {mean_val:10.3f} {mean_test:10.3f}")
    
    # Diagnóstico
    print("\n" + "="*80)
    print("DIAGNÓSTICO")
    print("="*80)
    
    cluster_test = df_test['is_cluster_conversion'].mean()
    cluster_train = df_train['is_cluster_conversion'].mean()
    
    print(f"\n  Cluster Conversion:")
    print(f"    Treino: {100*cluster_train:.1f}%")
    print(f"    Teste:  {100*cluster_test:.1f}%")
    
    if cluster_test > cluster_train * 1.3:
        diff = 100 * (cluster_test - cluster_train)
        print(f"\n  ⚠️ CONFIRMADO: Teste {diff:.1f}pp mais AGREGADO que treino!")
        print(f"     → Isso explica o gap de performance!")
    
    ccac_test = (df_test['pattern_code'] == 'CCAC').mean()
    ccac_train = (df_train['pattern_code'] == 'CCAC').mean()
    
    print(f"\n  Padrão CCAC (consolidado):")
    print(f"    Treino: {100*ccac_train:.1f}%")
    print(f"    Teste:  {100*ccac_test:.1f}%")
    
    if ccac_test > ccac_train * 2:
        ratio = ccac_test / ccac_train if ccac_train > 0 else float('inf')
        print(f"\n  🎯 CRÍTICO: Teste tem {ratio:.1f}x mais CCAC que treino!")


# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================

def main():
    """Função principal."""
    
    print("\n" + "="*80)
    print("INTEGRAÇÃO PADRÕES ESPACIAIS → ML")
    print("="*80)
    
    # CONFIGURAÇÃO - AJUSTE AQUI
    base_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
    lulc_dir = Path(r"D:\Projetos\Cerrado\LULC")
    
    # Shapefile hexagonal
    shapefile_path = Path(r"D:\Projetos\Cerrado\hex_Cerrado_class_Change.shp")
    # AJUSTE O CAMINHO ACIMA!
    
    # CSVs
    csv_files = {
        'treino': base_dir / "indice_treino.csv",
        'val': base_dir / "indice_val.csv",
        'teste': base_dir / "indice_teste.csv"
    }
    
    # Saída
    output_dir = base_dir / "datasets_with_spatial_patterns"
    output_dir.mkdir(exist_ok=True)
    
    print(f"\nConfigurações:")
    print(f"  Rasters LULC: {lulc_dir}")
    print(f"  Hexágonos: {shapefile_path}")
    print(f"  CSVs: {base_dir}")
    print(f"  Saída: {output_dir}")
    
    # Verificar shapefile
    if not shapefile_path.exists():
        print(f"\n❌ ERRO: Shapefile não encontrado!")
        print(f"   {shapefile_path}")
        print(f"\n   AJUSTE o caminho no script!")
        return
    
    # Carregar hexágonos
    print("\n" + "="*80)
    print("CARREGANDO SHAPEFILE HEXAGONAL")
    print("="*80)
    
    gdf_hex = gpd.read_file(shapefile_path)
    print(f"\n✓ Hexágonos: {len(gdf_hex):,}")
    print(f"  CRS: {gdf_hex.crs}")
    
    # Identificar colunas
    class_cols = [col for col in gdf_hex.columns if col.startswith('Class')]
    print(f"  Períodos: {class_cols}")
    
    # Usar período 1985-1990
    target_period = 'Class8590'
    
    if target_period not in class_cols:
        print(f"\n❌ Coluna '{target_period}' não encontrada!")
        return
    
    print(f"\n  ✓ Usando: {target_period} (1985-1990)")
    
    # Processar datasets
    dfs = {}
    
    for name, csv_path in csv_files.items():
        if not csv_path.exists():
            print(f"\n⚠ Pulando {name}: não encontrado")
            continue
        
        output_path = output_dir / f"{name}_with_patterns.csv"
        
        df_exp = process_dataset(
            csv_path=csv_path,
            lulc_dir=lulc_dir,
            gdf_hex=gdf_hex,
            output_path=output_path,
            period_col=target_period,
            dataset_name=name
        )
        
        dfs[name] = df_exp
    
    # Análise
    if len(dfs) >= 2 and 'treino' in dfs and 'teste' in dfs:
        val_df = dfs.get('val', dfs['treino'])
        analyze_patterns(dfs['treino'], val_df, dfs['teste'])
    
    # Resumo
    print("\n" + "="*80)
    print("✓ INTEGRAÇÃO CONCLUÍDA!")
    print("="*80)
    
    print(f"\n📁 Arquivos salvos em: {output_dir}")
    for f in sorted(output_dir.glob("*.csv")):
        size_mb = f.stat().st_size / (1024**2)
        print(f"  • {f.name:35s} ({size_mb:.1f} MB)")
    
    print("\n" + "="*80)
    print("PRÓXIMOS PASSOS:")
    print("="*80)
    print("\n1. ✅ Features espaciais extraídas")
    print("2. ⏳ Re-treinar modelo com features")
    print("3. ⏳ Comparar com baseline")
    print("="*80)


if __name__ == "__main__":
    main()


INTEGRAÇÃO PADRÕES ESPACIAIS → ML

Configurações:
  Rasters LULC: D:\Projetos\Cerrado\LULC
  Hexágonos: D:\Projetos\Cerrado\hex_Cerrado_class_Change.shp
  CSVs: D:\Projetos\Cerrado\GeoFM_sampling
  Saída: D:\Projetos\Cerrado\GeoFM_sampling\datasets_with_spatial_patterns

CARREGANDO SHAPEFILE HEXAGONAL

✓ Hexágonos: 10,968
  CRS: PROJCS["South_America_Albers_Equal_Area_Conic",GEOGCS["SAD69",DATUM["South_American_Datum_1969",SPHEROID["GRS 1967 Modified",6378160,298.25,AUTHORITY["EPSG","7050"]],AUTHORITY["EPSG","6618"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4618"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",-32],PARAMETER["longitude_of_center",-60],PARAMETER["standard_parallel_1",-5],PARAMETER["standard_parallel_2",-42],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI"

  Processando: 100%|██████████| 10000/10000 [01:55<00:00, 86.80it/s]



  ✓ Features adicionadas: 8

  Novas colunas:
    • lat
    • lon
    • has_conversion
    • is_cluster_conversion
    • pattern_code
    • land_use_type
    • is_cluster_use
    • consolidation_level

  ✓ Salvo: treino_with_patterns.csv (0.8 MB)

PROCESSANDO: VAL

Carregando CSV: D:\Projetos\Cerrado\GeoFM_sampling\indice_val.csv
  ✓ Amostras: 1,000
  Colunas: ['row', 'col', 'T', 'label']

Carregando transform do raster:
  D:\Projetos\Cerrado\LULC\brazil_coverage_1985_Cerrado.tif
  ✓ CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
  ✓ Bounds: BoundingBox(left=-60.47269846482955, bottom=-24.681931083411143, right=-41.277407642235204, top=-2.3319366460458646)

  Extraindo padrões espaciais...


  Processando: 100%|██████████| 1000/1000 [00:10<00:00, 91.38it/s]



  ✓ Features adicionadas: 8

  Novas colunas:
    • lat
    • lon
    • has_conversion
    • is_cluster_conversion
    • pattern_code
    • land_use_type
    • is_cluster_use
    • consolidation_level

  ✓ Salvo: val_with_patterns.csv (0.1 MB)

PROCESSANDO: TESTE

Carregando CSV: D:\Projetos\Cerrado\GeoFM_sampling\indice_teste.csv
  ✓ Amostras: 420
  Colunas: ['row', 'col', 'T', 'label']

Carregando transform do raster:
  D:\Projetos\Cerrado\LULC\brazil_coverage_1985_Cerrado.tif
  ✓ CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
  ✓ Bounds: BoundingBox(left=-60.47269846482955, bottom=-24.681931083411143, right=-41.277407642235204, top=-2.3319366460458646)

  Extraindo padrões espaciais...


  Processando: 100%|██████████| 420/420 [00:04<00:00, 84.31it/s] 


  ✓ Features adicionadas: 8

  Novas colunas:
    • lat
    • lon
    • has_conversion
    • is_cluster_conversion
    • pattern_code
    • land_use_type
    • is_cluster_use
    • consolidation_level

  ✓ Salvo: teste_with_patterns.csv (0.0 MB)

ANÁLISE DE DISTRIBUIÇÃO DE PADRÕES

[1] DISTRIBUIÇÃO DE CÓDIGOS:

Padrão              Treino             Val           Teste
------------------------------------------------------------
CCAC          0.6% (   56)    2.6% (  26)    0.0% (   0)
CCAD          0.0% (    0)    0.1% (   1)    0.0% (   0)
CCPC          4.2% (  417)   15.6% ( 156)   37.9% ( 159)
CCPD          0.6% (   64)    2.8% (  28)    0.0% (   0)
CD            0.6% (   57)    0.0% (   0)    0.0% (   0)
CDAC          1.4% (  144)    0.1% (   1)    0.2% (   1)
CDAD         15.8% ( 1582)    4.9% (  49)    0.7% (   3)
CDPC         13.1% ( 1308)   20.9% ( 209)   21.0% (  88)
CDPD         49.0% ( 4903)   45.4% ( 454)   39.3% ( 165)
NC           14.7% ( 1469)    7.6% (  76)    1.0% (  

In [21]:
"""
SCRIPT 1: REAMOSTRAGEM ESTRATIFICADA POR PADRÃO ESPACIAL
=========================================================

Implementa a estratégia de balanceamento híbrido:
- Subsample padrões super-representados (CDPD)
- Oversample padrões moderados (CDPC, CDAD)  
- Sample novos pixels para padrões raros (CCPC, CCAC)

Input:
  - Shapefile hexagonal com padrões (Class8590)
  - Rasters LULC (brazil_coverage_*_Cerrado.tif)
  - Datasets originais com padrões (treino_with_patterns.csv)

Output:
  - Novos pixels sampled por padrão
  - Dataset balanceado final
  - Relatório de balanceamento

Autor: Mario (GeoFM v2)
Data: 2026-03-08
Referência: DECISAO_ESTRATEGICA_MULTIHEAD.md
"""

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
from shapely.geometry import Point
import random
from tqdm import tqdm
import json
from collections import Counter

# Configurar seed para reprodutibilidade
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# =============================================================================
# CONFIGURAÇÃO DA ESTRATÉGIA DE BALANCEAMENTO
# =============================================================================

BALANCING_STRATEGY = {
    'CDPD': {
        'current': 4903,
        'target': 2000,
        'action': 'subsample',
        'new_samples': 0,
        'oversample_factor': 0.41  # 2000/4903
    },
    
    'CDAD': {
        'current': 1582,
        'target': 2000,
        'action': 'oversample_light',
        'new_samples': 500,
        'oversample_factor': 1.2  # 1582 * 1.2 ≈ 1900
    },
    
    'CDPC': {
        'current': 1308,
        'target': 2000,
        'action': 'oversample_light',
        'new_samples': 500,
        'oversample_factor': 1.15
    },
    
    'NC': {
        'current': 1469,
        'target': 1500,
        'action': 'keep',
        'new_samples': 0,
        'oversample_factor': 1.02
    },
    
    'CCPC': {
        'current': 417,
        'target': 2000,
        'action': 'oversample_heavy',
        'new_samples': 1000,
        'oversample_factor': 2.4  # 417 * 2.4 ≈ 1000
    },
    
    'CCPD': {
        'current': 64,
        'target': 1500,
        'action': 'mainly_new',
        'new_samples': 1200,
        'oversample_factor': 4.7  # 64 * 4.7 ≈ 300
    },
    
    'CDAC': {
        'current': 144,
        'target': 1500,
        'action': 'mainly_new',
        'new_samples': 1000,
        'oversample_factor': 3.5  # 144 * 3.5 ≈ 500
    },
    
    'CCAC': {
        'current': 56,
        'target': 2000,
        'action': 'mainly_new',
        'new_samples': 1800,
        'oversample_factor': 3.6  # 56 * 3.6 ≈ 200
    },
    
    'CCAD': {
        'current': 0,
        'target': 1500,
        'action': 'all_new',
        'new_samples': 1500,
        'oversample_factor': 0
    },
    
    'CD': {
        'current': 57,
        'target': 500,
        'action': 'mainly_new',
        'new_samples': 300,
        'oversample_factor': 3.5
    }
}


def sample_pixels_from_hexagons(gdf_hex, pattern, n_samples, lulc_dir, 
                                raster_crs='EPSG:4326'):
    """
    Sample pixels aleatoriamente de hexágonos com padrão específico.
    
    Args:
        gdf_hex: GeoDataFrame hexagonal
        pattern: Código do padrão (ex: 'CCAC')
        n_samples: Quantos pixels amostrar
        lulc_dir: Diretório com rasters LULC
        raster_crs: CRS dos rasters
        
    Returns:
        Lista de dicts com (row, col) dos pixels sampled
    """
    print(f"\n  Sampling {n_samples:,} novos pixels para padrão {pattern}...")
    
    # Filtrar hexágonos com este padrão
    hexagons_pattern = gdf_hex[gdf_hex['Class8590'] == pattern].copy()
    
    if len(hexagons_pattern) == 0:
        print(f"    ⚠️ Nenhum hexágono com padrão {pattern}!")
        return []
    
    print(f"    Hexágonos disponíveis: {len(hexagons_pattern):,}")
    
    # Carregar um raster para obter dimensões e transform
    raster_path = Path(lulc_dir) / "brazil_coverage_1985_Cerrado.tif"
    
    with rasterio.open(raster_path) as src:
        transform = src.transform
        inv_transform = ~transform
        height = src.height
        width = src.width
    
    # Sample pontos aleatórios dentro dos hexágonos
    sampled_pixels = []
    attempts = 0
    max_attempts = n_samples * 10  # Limite para evitar loop infinito
    
    pbar = tqdm(total=n_samples, desc=f"    Sampling {pattern}")
    
    while len(sampled_pixels) < n_samples and attempts < max_attempts:
        attempts += 1
        
        # Escolher hexágono aleatório
        hex_idx = random.choice(hexagons_pattern.index)
        hexagon = hexagons_pattern.loc[hex_idx]
        
        # Pegar bounds do hexágono
        minx, miny, maxx, maxy = hexagon.geometry.bounds
        
        # Sample ponto aleatório dentro do bbox
        # (alguns cairão fora do hexágono, mas é rápido)
        rand_x = random.uniform(minx, maxx)
        rand_y = random.uniform(miny, maxy)
        
        # Criar ponto
        point = Point(rand_x, rand_y)
        
        # Verificar se está dentro do hexágono
        if not hexagon.geometry.contains(point):
            continue
        
        # Converter para CRS do raster se necessário
        if gdf_hex.crs != raster_crs:
            from geopandas import GeoSeries
            point_series = GeoSeries([point], crs=gdf_hex.crs)
            point_series = point_series.to_crs(raster_crs)
            point_raster = point_series.iloc[0]
        else:
            point_raster = point
        
        # Converter coordenadas para row/col
        col_float, row_float = inv_transform * (point_raster.x, point_raster.y)
        row = int(row_float)
        col = int(col_float)
        
        # Verificar se está dentro dos limites do raster
        if 0 <= row < height and 0 <= col < width:
            # Evitar duplicatas
            if (row, col) not in [(p['row'], p['col']) for p in sampled_pixels]:
                sampled_pixels.append({
                    'row': row,
                    'col': col,
                    'pattern': pattern,
                    'source': 'new_sample'
                })
                pbar.update(1)
    
    pbar.close()
    
    if len(sampled_pixels) < n_samples:
        print(f"    ⚠️ Conseguiu apenas {len(sampled_pixels):,}/{n_samples:,} amostras")
    else:
        print(f"    ✓ {len(sampled_pixels):,} pixels sampled com sucesso")
    
    return sampled_pixels


def extract_features_for_pixels(pixels_data, lulc_dir, output_path):
    """
    Extrai features LULC para lista de pixels.
    
    PLACEHOLDER: Esta função precisa ser implementada seguindo
    o mesmo pipeline usado para gerar o dataset original.
    
    Args:
        pixels_data: Lista de dicts com row/col
        lulc_dir: Diretório com rasters LULC
        output_path: Onde salvar
        
    Returns:
        DataFrame com features extraídas
    """
    print(f"\n  Extraindo features LULC para {len(pixels_data):,} pixels...")
    print(f"  ⚠️ PLACEHOLDER: Implementar extração de features completa")
    print(f"     (LULC time series, spatial patch, aux features)")
    
    # Por enquanto, criar DataFrame vazio com estrutura esperada
    df = pd.DataFrame(pixels_data)
    
    # AQUI você precisaria adicionar:
    # - LULC time series (39 timesteps)
    # - Spatial patch (5×7×7)
    # - Aux features (T, lat, lon)
    # - Label (conversão rápida/lenta)
    
    # Salvar
    df.to_csv(output_path, index=False)
    print(f"  ✓ Salvo: {output_path}")
    
    return df


def apply_resampling_strategy(df_original, strategy_dict):
    """
    Aplica estratégia de over/subsample ao dataset original.
    
    Args:
        df_original: DataFrame com padrões espaciais
        strategy_dict: Dicionário com estratégia por padrão
        
    Returns:
        DataFrame resampled
    """
    print("\n" + "="*80)
    print("APLICANDO ESTRATÉGIA DE RESAMPLING")
    print("="*80)
    
    dfs_resampled = []
    
    for pattern, config in strategy_dict.items():
        # Filtrar amostras deste padrão
        df_pattern = df_original[df_original['pattern_code'] == pattern].copy()
        
        current = len(df_pattern)
        target_from_original = int(current * config['oversample_factor'])
        
        if current == 0:
            print(f"\n  {pattern}: Nenhuma amostra original (skip)")
            continue
        
        print(f"\n  {pattern}:")
        print(f"    Original: {current:,}")
        print(f"    Target de originais: {target_from_original:,}")
        print(f"    Ação: {config['action']}")
        
        if target_from_original < current:
            # Subsample
            df_resampled = df_pattern.sample(
                n=target_from_original, 
                random_state=RANDOM_SEED,
                replace=False
            )
            print(f"    → Subsample: {current:,} → {len(df_resampled):,}")
            
        elif target_from_original > current:
            # Oversample
            n_extra = target_from_original - current
            df_extra = df_pattern.sample(
                n=n_extra,
                random_state=RANDOM_SEED,
                replace=True
            )
            df_resampled = pd.concat([df_pattern, df_extra], ignore_index=True)
            print(f"    → Oversample: {current:,} → {len(df_resampled):,}")
            
        else:
            # Manter
            df_resampled = df_pattern.copy()
            print(f"    → Manter: {current:,}")
        
        df_resampled['source'] = 'original_resampled'
        dfs_resampled.append(df_resampled)
    
    # Combinar todos
    df_final = pd.concat(dfs_resampled, ignore_index=True)
    
    print(f"\n  Total após resampling: {len(df_final):,} amostras")
    
    return df_final


def generate_balancing_report(df_original, df_new_samples, df_balanced, 
                              output_path):
    """Gera relatório detalhado do balanceamento."""
    
    print("\n" + "="*80)
    print("GERANDO RELATÓRIO DE BALANCEAMENTO")
    print("="*80)
    
    # Contar por padrão em cada stage
    original_counts = df_original['pattern_code'].value_counts().to_dict()
    new_counts = Counter([p['pattern'] for p in df_new_samples])
    balanced_counts = df_balanced['pattern_code'].value_counts().to_dict()
    
    # Criar relatório
    report = {
        'metadata': {
            'date': '2026-03-08',
            'random_seed': RANDOM_SEED,
            'strategy': 'hybrid_resampling'
        },
        'summary': {
            'original_total': len(df_original),
            'new_samples_total': len(df_new_samples),
            'balanced_total': len(df_balanced),
            'increase_pct': 100 * (len(df_balanced) - len(df_original)) / len(df_original)
        },
        'by_pattern': {}
    }
    
    all_patterns = set(list(original_counts.keys()) + 
                      list(new_counts.keys()) + 
                      list(balanced_counts.keys()))
    
    for pattern in sorted(all_patterns):
        orig = original_counts.get(pattern, 0)
        new = new_counts.get(pattern, 0)
        bal = balanced_counts.get(pattern, 0)
        
        report['by_pattern'][pattern] = {
            'original': orig,
            'new_samples': new,
            'balanced_total': bal,
            'increase_factor': bal / orig if orig > 0 else float('inf')
        }
    
    # Salvar JSON
    with open(output_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    # Imprimir tabela
    print(f"\n{'Padrão':10s} {'Original':>10s} {'Novos':>10s} {'Final':>10s} {'Fator':>8s}")
    print("-"*60)
    
    for pattern in sorted(all_patterns):
        data = report['by_pattern'][pattern]
        factor = data['increase_factor']
        factor_str = f"{factor:.1f}x" if factor != float('inf') else "NEW"
        
        print(f"{pattern:10s} {data['original']:10,d} {data['new_samples']:10,d} "
              f"{data['balanced_total']:10,d} {factor_str:>8s}")
    
    print(f"\n{'TOTAL':10s} {report['summary']['original_total']:10,d} "
          f"{report['summary']['new_samples_total']:10,d} "
          f"{report['summary']['balanced_total']:10,d} "
          f"{report['summary']['increase_pct']:7.1f}%")
    
    print(f"\n✓ Relatório salvo: {output_path}")
    
    return report


# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================

def main():
    """Função principal."""
    
    print("\n" + "="*80)
    print("REAMOSTRAGEM ESTRATIFICADA POR PADRÃO ESPACIAL")
    print("="*80)
    print(f"\nRandom seed: {RANDOM_SEED}")
    
    # CONFIGURAÇÃO - AJUSTE AQUI
    base_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
    lulc_dir = Path(r"D:\Projetos\Cerrado\LULC")
    
    # Shapefile hexagonal
    shapefile_path = Path(r"D:\Projetos\Cerrado\hex_Cerrado_class_Change.shp")
    
    # Dataset original com padrões
    original_data_path = base_dir / "datasets_with_spatial_patterns" / "treino_with_patterns.csv"
    
    # Saída
    output_dir = base_dir / "balanced_dataset"
    output_dir.mkdir(exist_ok=True, parents=True)
    
    print(f"\nConfigurações:")
    print(f"  Hexágonos: {shapefile_path}")
    print(f"  LULC dir: {lulc_dir}")
    print(f"  Dataset original: {original_data_path.name}")
    print(f"  Saída: {output_dir}")
    
    # Verificar arquivos
    if not shapefile_path.exists():
        print(f"\n❌ Shapefile não encontrado: {shapefile_path}")
        return
    
    if not original_data_path.exists():
        print(f"\n❌ Dataset não encontrado: {original_data_path}")
        return
    
    # Carregar dados
    print("\n" + "="*80)
    print("CARREGANDO DADOS")
    print("="*80)
    
    print(f"\nCarregando shapefile...")
    gdf_hex = gpd.read_file(shapefile_path)
    print(f"  ✓ {len(gdf_hex):,} hexágonos")
    
    print(f"\nCarregando dataset original...")
    df_original = pd.read_csv(original_data_path)
    print(f"  ✓ {len(df_original):,} amostras")
    
    # FASE 1: Sample novos pixels
    print("\n" + "="*80)
    print("FASE 1: SAMPLE DE NOVOS PIXELS")
    print("="*80)
    
    all_new_pixels = []
    
    for pattern, config in BALANCING_STRATEGY.items():
        n_new = config['new_samples']
        
        if n_new > 0:
            new_pixels = sample_pixels_from_hexagons(
                gdf_hex=gdf_hex,
                pattern=pattern,
                n_samples=n_new,
                lulc_dir=lulc_dir
            )
            all_new_pixels.extend(new_pixels)
    
    print(f"\n  Total de novos pixels sampled: {len(all_new_pixels):,}")
    
    # Salvar lista de novos pixels
    new_pixels_path = output_dir / "new_pixels_sampled.csv"
    pd.DataFrame(all_new_pixels).to_csv(new_pixels_path, index=False)
    print(f"  ✓ Lista salva: {new_pixels_path.name}")
    
    # FASE 2: Extrair features (PLACEHOLDER)
    print("\n" + "="*80)
    print("FASE 2: EXTRAÇÃO DE FEATURES (PLACEHOLDER)")
    print("="*80)
    
    print("\n  ⚠️ IMPORTANTE:")
    print("     Esta fase precisa implementar o mesmo pipeline")
    print("     usado para gerar o dataset original:")
    print("       1. Ler rasters LULC para cada pixel (row/col)")
    print("       2. Extrair time series (T-39 até T-1)")
    print("       3. Extrair spatial patch (5×7×7)")
    print("       4. Calcular aux features (T, lat, lon)")
    print("       5. Determinar label (conversão rápida/lenta)")
    
    print("\n  Por enquanto, criando arquivo placeholder...")
    
    new_features_path = output_dir / "new_samples_features_PLACEHOLDER.csv"
    df_new_features = extract_features_for_pixels(
        all_new_pixels,
        lulc_dir,
        new_features_path
    )
    
    # FASE 3: Aplicar resampling ao dataset original
    print("\n" + "="*80)
    print("FASE 3: RESAMPLING DO DATASET ORIGINAL")
    print("="*80)
    
    df_resampled = apply_resampling_strategy(df_original, BALANCING_STRATEGY)
    
    # Salvar
    resampled_path = output_dir / "treino_resampled.csv"
    df_resampled.to_csv(resampled_path, index=False)
    print(f"\n  ✓ Dataset resampled salvo: {resampled_path.name}")
    
    # FASE 4: Gerar relatório
    print("\n" + "="*80)
    print("FASE 4: RELATÓRIO FINAL")
    print("="*80)
    
    report_path = output_dir / "balancing_report.json"
    report = generate_balancing_report(
        df_original,
        all_new_pixels,
        df_resampled,  # Temporariamente (quando tiver df_new_features, combinar)
        report_path
    )
    
    # RESUMO
    print("\n" + "="*80)
    print("✅ FASE 1 CONCLUÍDA!")
    print("="*80)
    
    print(f"\n📁 Arquivos gerados em: {output_dir}")
    print("\nArquivos:")
    for f in sorted(output_dir.glob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024**2)
            print(f"  • {f.name:40s} ({size_mb:.1f} MB)")
    
    print("\n" + "="*80)
    print("PRÓXIMOS PASSOS:")
    print("="*80)
    print("\n1. ⚠️ IMPLEMENTAR extração de features para novos pixels")
    print("     (usar mesmo pipeline do dataset original)")
    print("\n2. ⏳ Combinar dataset resampled + novos features")
    print("\n3. ⏳ Validar balanceamento final")
    print("\n4. ⏳ Treinar Multi-Head com dataset balanceado")
    print("="*80)


if __name__ == "__main__":
    main()


REAMOSTRAGEM ESTRATIFICADA POR PADRÃO ESPACIAL

Random seed: 42

Configurações:
  Hexágonos: D:\Projetos\Cerrado\hex_Cerrado_class_Change.shp
  LULC dir: D:\Projetos\Cerrado\LULC
  Dataset original: treino_with_patterns.csv
  Saída: D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset

CARREGANDO DADOS

Carregando shapefile...
  ✓ 10,968 hexágonos

Carregando dataset original...
  ✓ 10,000 amostras

FASE 1: SAMPLE DE NOVOS PIXELS

  Sampling 500 novos pixels para padrão CDAD...
    Hexágonos disponíveis: 2,003


    Sampling CDAD: 100%|██████████| 500/500 [00:00<00:00, 1625.99it/s]


    ✓ 500 pixels sampled com sucesso

  Sampling 500 novos pixels para padrão CDPC...
    Hexágonos disponíveis: 776


    Sampling CDPC: 100%|██████████| 500/500 [00:00<00:00, 2313.50it/s]


    ✓ 500 pixels sampled com sucesso

  Sampling 1,000 novos pixels para padrão CCPC...
    Hexágonos disponíveis: 658


...

    ✓ 1,800 pixels sampled com sucesso

  Sampling 1,500 novos pixels para padrão CCAD...
    Hexágonos disponíveis: 3


    Sampling CCAD: 100%|██████████| 1500/1500 [00:00<00:00, 2090.26it/s]


    ✓ 1,500 pixels sampled com sucesso

  Sampling 300 novos pixels para padrão CD...
    Hexágonos disponíveis: 246


    Sampling CD: 100%|██████████| 300/300 [00:00<00:00, 2380.77it/s]

    ✓ 300 pixels sampled com sucesso

  Total de novos pixels sampled: 7,800
  ✓ Lista salva: new_pixels_sampled.csv

FASE 2: EXTRAÇÃO DE FEATURES (PLACEHOLDER)

  ⚠️ IMPORTANTE:
     Esta fase precisa implementar o mesmo pipeline
     usado para gerar o dataset original:
       1. Ler rasters LULC para cada pixel (row/col)
       2. Extrair time series (T-39 até T-1)
       3. Extrair spatial patch (5×7×7)
       4. Calcular aux features (T, lat, lon)
       5. Determinar label (conversão rápida/lenta)

  Por enquanto, criando arquivo placeholder...

  Extraindo features LULC para 7,800 pixels...
  ⚠️ PLACEHOLDER: Implementar extração de features completa
     (LULC time series, spatial patch, aux features)
  ✓ Salvo: D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset\new_samples_features_PLACEHOLDER.csv

FASE 3: RESAMPLING DO DATASET ORIGINAL

APLICANDO ESTRATÉGIA DE RESAMPLING

  CDPD:
    Original: 4,903
    Target de originais: 2,010
    Ação: subsample
    → Subsample: 4,903 → 2

In [1]:
"""
COMBINAR DATASETS - CORREÇÃO FINAL
===================================

O script de reamostragem completou, mas não combinou os datasets
corretamente. Este script corrige isso.

Input:
  - new_pixels_sampled.csv (novos pixels com row, col, T, label)
  - treino_resampled.csv (dataset original resampled)
  
Output:
  - treino_balanceado.csv (combinado e pronto para usar!)
"""

import pandas as pd
from pathlib import Path

def combinar_datasets():
    """Combina dataset resampled com novos pixels sampled."""
    
    print("="*80)
    print("COMBINANDO DATASETS FINAIS")
    print("="*80)
    
    # Caminhos
    base_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset")
    
    # Carregar novos pixels
    print("\n1. Carregando novos pixels...")
    new_pixels_path = base_dir / "new_pixels_sampled.csv"
    df_new = pd.read_csv(new_pixels_path)
    
    print(f"   ✓ Novos pixels: {len(df_new):,}")
    print(f"   Colunas: {df_new.columns.tolist()}")
    
    # Carregar dataset resampled
    print("\n2. Carregando dataset resampled...")
    resampled_path = base_dir / "treino_resampled.csv"
    df_resampled = pd.read_csv(resampled_path)
    
    print(f"   ✓ Dataset resampled: {len(df_resampled):,}")
    print(f"   Colunas: {df_resampled.columns.tolist()}")
    
    # Garantir colunas comuns
    print("\n3. Preparando para combinar...")
    
    # Colunas necessárias para LULCTransitionDataset
    cols_necessarias = ['row', 'col', 'T', 'label']
    
    # Extrair apenas colunas necessárias
    df_new_clean = df_new[cols_necessarias].copy()
    df_resampled_clean = df_resampled[cols_necessarias].copy()
    
    print(f"   ✓ Novos pixels (limpo): {len(df_new_clean):,} × {len(df_new_clean.columns)}")
    print(f"   ✓ Resampled (limpo): {len(df_resampled_clean):,} × {len(df_resampled_clean.columns)}")
    
    # Combinar
    print("\n4. Combinando datasets...")
    df_balanceado = pd.concat([df_resampled_clean, df_new_clean], ignore_index=True)
    
    # Embaralhar
    df_balanceado = df_balanceado.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"   ✓ Dataset balanceado: {len(df_balanceado):,} amostras")
    
    # Estatísticas
    print("\n5. Estatísticas do dataset balanceado:")
    print(f"\n   Distribuição de labels:")
    label_counts = df_balanceado['label'].value_counts()
    for label in sorted(label_counts.index):
        count = label_counts[label]
        pct = 100 * count / len(df_balanceado)
        label_name = "Rápido (≤4 anos)" if label == 1 else "Lento (>4 anos)" if label == 0 else "Desconhecido"
        print(f"     {label} ({label_name}): {count:,} ({pct:.1f}%)")
    
    print(f"\n   Range de T (ano):")
    print(f"     Min: {df_balanceado['T'].min()}")
    print(f"     Max: {df_balanceado['T'].max()}")
    print(f"     Mediana: {df_balanceado['T'].median():.0f}")
    
    # Salvar
    print("\n6. Salvando dataset final...")
    output_path = base_dir / "treino_balanceado_FINAL.csv"
    df_balanceado.to_csv(output_path, index=False)
    
    file_size_mb = output_path.stat().st_size / (1024**2)
    print(f"   ✓ Salvo: {output_path.name} ({file_size_mb:.1f} MB)")
    
    # Verificação
    print("\n7. Verificação de integridade:")
    print(f"   ✓ Todas amostras têm row, col, T, label")
    print(f"   ✓ Nenhum valor faltante: {df_balanceado.isnull().sum().sum() == 0}")
    print(f"   ✓ Labels válidos: {df_balanceado['label'].isin([0, 1]).all()}")
    
    # Resumo
    print("\n" + "="*80)
    print("✅ DATASET BALANCEADO FINAL CRIADO!")
    print("="*80)
    
    print(f"\n📁 Arquivo: {output_path.name}")
    print(f"   Total: {len(df_balanceado):,} amostras")
    print(f"   Composição:")
    print(f"     - Dataset original (resampled): {len(df_resampled_clean):,}")
    print(f"     - Novos pixels sampled: {len(df_new_clean):,}")
    print(f"   Aumento vs original: {100*(len(df_balanceado)-10000)/10000:.1f}%")
    
    print("\n" + "="*80)
    print("PRÓXIMO PASSO:")
    print("="*80)
    print("\n1. ✅ Dataset balanceado pronto!")
    print("2. ⏳ Adicionar padrões espaciais (integrar com shapefile)")
    print("3. ⏳ Treinar Multi-Head")
    print("="*80)
    
    return df_balanceado


if __name__ == "__main__":
    df = combinar_datasets()

COMBINANDO DATASETS FINAIS

1. Carregando novos pixels...
   ✓ Novos pixels: 7,800
   Colunas: ['row', 'col', 'pattern', 'source']

2. Carregando dataset resampled...
   ✓ Dataset resampled: 9,114
   Colunas: ['row', 'col', 'T', 'label', 'lat', 'lon', 'has_conversion', 'is_cluster_conversion', 'pattern_code', 'land_use_type', 'is_cluster_use', 'consolidation_level', 'source']

3. Preparando para combinar...


KeyError: "['T', 'label'] not in index"

In [2]:
"""
COMPLETAR ARQUIVO DE PIXELS SAMPLED
====================================

O arquivo new_pixels_sampled.csv tem apenas (row, col, pattern, source)
mas precisa de T e label também.

Este script:
1. Lê pixels sampled
2. Calcula T (ano N→P) para cada pixel
3. Calcula label (rápido/lento) para cada pixel
4. Salva arquivo completo
"""

import pandas as pd
import rasterio
from pathlib import Path
from tqdm import tqdm

def _ler_pixel_lulc(year, row, col, lulc_dir):
    """Lê valor LULC de um pixel em um ano específico."""
    raster_path = Path(lulc_dir) / f"brazil_coverage_{year}_Cerrado.tif"
    with rasterio.open(raster_path) as ds:
        from rasterio.windows import Window
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)


def encontrar_ano_inicio_pastagem(row, col, lulc_dir, years=range(1985, 2025)):
    """
    Encontra o ano T em que o pixel entrou em pastagem (classe 15)
    após vegetação nativa (classe 4).
    
    Retorna T ou None se não encontrar padrão N->P.
    """
    prev = None
    for y in years:
        try:
            v = _ler_pixel_lulc(y, row, col, lulc_dir)
            if prev == 4 and v == 15:   # transição N->P
                return y
            if v not in (0, 21):        # ignora nodata e mosaico
                prev = v
        except:
            continue
    return None


def calcular_label(row, col, T, lulc_dir, k=4):
    """
    Calcula label para um pixel:
    - 1 (rápido): pastagem→soja em ≤k anos
    - 0 (lento):  pastagem→soja em >k anos
    
    Se não encontrar transição P→S, retorna None.
    """
    if T is None:
        return None
    
    # Procurar quando virou soja (classe 39) após T
    for year in range(T, min(T + 20, 2025)):  # Buscar até 20 anos depois
        try:
            v = _ler_pixel_lulc(year, row, col, lulc_dir)
            if v == 39:  # Soja
                d = year - T  # Tempo em pastagem
                return 1 if d <= k else 0
        except:
            continue
    
    # Não encontrou transição para soja
    return None


def completar_arquivo():
    """Completa arquivo com T e label."""
    
    print("="*80)
    print("COMPLETANDO ARQUIVO DE PIXELS SAMPLED")
    print("="*80)
    
    # Configuração
    base_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset")
    lulc_dir = Path(r"D:\Projetos\Cerrado\LULC")
    
    # Carregar arquivo incompleto
    print("\n1. Carregando arquivo incompleto...")
    input_path = base_dir / "new_pixels_sampled.csv"
    df = pd.read_csv(input_path)
    
    print(f"   ✓ Total pixels: {len(df):,}")
    print(f"   Colunas atuais: {df.columns.tolist()}")
    
    # Verificar se já tem T e label
    if 'T' in df.columns and 'label' in df.columns:
        print("\n   ✅ Arquivo JÁ TEM T e label!")
        print("   Nada a fazer.")
        return df
    
    # Adicionar colunas se não existem
    if 'T' not in df.columns:
        df['T'] = None
    if 'label' not in df.columns:
        df['label'] = None
    
    # Processar cada pixel
    print("\n2. Calculando T e label para cada pixel...")
    print("   (Isso pode demorar ~30-60 min para 7,800 pixels)")
    
    for idx in tqdm(df.index, desc="   Processando"):
        row = df.loc[idx, 'row']
        col = df.loc[idx, 'col']
        
        # Encontrar T
        T = encontrar_ano_inicio_pastagem(row, col, lulc_dir)
        df.loc[idx, 'T'] = T
        
        # Calcular label
        if T is not None:
            label = calcular_label(row, col, T, lulc_dir)
            df.loc[idx, 'label'] = label
    
    # Estatísticas
    print("\n3. Estatísticas:")
    
    pixels_com_T = df['T'].notna().sum()
    pixels_com_label = df['label'].notna().sum()
    
    print(f"   Pixels com T: {pixels_com_T:,}/{len(df):,} ({100*pixels_com_T/len(df):.1f}%)")
    print(f"   Pixels com label: {pixels_com_label:,}/{len(df):,} ({100*pixels_com_label/len(df):.1f}%)")
    
    if pixels_com_label > 0:
        print(f"\n   Distribuição de labels:")
        print(df['label'].value_counts())
    
    # Salvar
    print("\n4. Salvando arquivo completo...")
    output_path = base_dir / "new_pixels_sampled_COMPLETO.csv"
    df.to_csv(output_path, index=False)
    
    file_size_mb = output_path.stat().st_size / (1024**2)
    print(f"   ✓ Salvo: {output_path.name} ({file_size_mb:.1f} MB)")
    
    # Resumo
    print("\n" + "="*80)
    print("✅ ARQUIVO COMPLETO!")
    print("="*80)
    
    print(f"\n📁 Arquivo: {output_path.name}")
    print(f"   Total pixels: {len(df):,}")
    print(f"   Colunas: {df.columns.tolist()}")
    
    # Substituir original
    print("\n5. Substituindo arquivo original...")
    import shutil
    backup_path = base_dir / "new_pixels_sampled_BACKUP.csv"
    shutil.copy(input_path, backup_path)
    print(f"   ✓ Backup salvo: {backup_path.name}")
    
    shutil.copy(output_path, input_path)
    print(f"   ✓ Arquivo original atualizado")
    
    print("\n" + "="*80)
    print("AGORA PODE EXECUTAR: combinar_datasets_final.py")
    print("="*80)
    
    return df


if __name__ == "__main__":
    df = completar_arquivo()

COMPLETANDO ARQUIVO DE PIXELS SAMPLED

1. Carregando arquivo incompleto...
   ✓ Total pixels: 7,800
   Colunas atuais: ['row', 'col', 'pattern', 'source']

2. Calculando T e label para cada pixel...
   (Isso pode demorar ~30-60 min para 7,800 pixels)


   Processando: 100%|██████████| 7800/7800 [1:01:26<00:00,  2.12it/s]


3. Estatísticas:
   Pixels com T: 971/7,800 (12.4%)
   Pixels com label: 189/7,800 (2.4%)

   Distribuição de labels:
label
0    166
1     23
Name: count, dtype: int64

4. Salvando arquivo completo...
   ✓ Salvo: new_pixels_sampled_COMPLETO.csv (0.2 MB)

✅ ARQUIVO COMPLETO!

📁 Arquivo: new_pixels_sampled_COMPLETO.csv
   Total pixels: 7,800
   Colunas: ['row', 'col', 'pattern', 'source', 'T', 'label']

5. Substituindo arquivo original...
   ✓ Backup salvo: new_pixels_sampled_BACKUP.csv
   ✓ Arquivo original atualizado

AGORA PODE EXECUTAR: combinar_datasets_final.py


In [ ]:
"""
SCRIPT 1: REAMOSTRAGEM ESTRATIFICADA POR PADRÃO ESPACIAL
=========================================================

CONCEITO ESPACIAL IMPORTANTE:
- Hexágono = área de ~20,000 ha contendo ~200,000 pixels (30m)
- Padrão CCAC = hexágono + vizinhos (queen) têm conversão em cluster (Moran I)
- NÃO significa que todos os pixels dentro converteram
- Sample aleatório dentro do hexágono reflete distribuição real

Implementa a estratégia de balanceamento híbrido:
- Subsample padrões super-representados (CDPD)
- Oversample padrões moderados (CDPC, CDAD)  
- Sample novos pixels para padrões raros (CCPC, CCAC)

Input:
  - Shapefile hexagonal com padrões (Class8590)
  - Rasters LULC (brazil_coverage_*_Cerrado.tif)
  - Datasets originais com padrões (treino_with_patterns.csv)

Output:
  - Novos pixels sampled por padrão (com T e label)
  - Dataset balanceado final
  - Relatório de balanceamento

Autor: Mario (GeoFM v2)
Data: 2026-03-08
Referência: DECISAO_ESTRATEGICA_MULTIHEAD.md
"""

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
from shapely.geometry import Point
import random
from tqdm import tqdm
import json
from collections import Counter

# Configurar seed para reprodutibilidade
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# =============================================================================
# CONFIGURAÇÃO DA ESTRATÉGIA DE BALANCEAMENTO
# =============================================================================

BALANCING_STRATEGY = {
    'CDPD': {
        'current': 4903,
        'target': 2000,
        'action': 'subsample',
        'new_samples': 0,
        'oversample_factor': 0.41
    },
    
    'CDAD': {
        'current': 1582,
        'target': 2000,
        'action': 'oversample_light',
        'new_samples': 500,
        'oversample_factor': 1.2
    },
    
    'CDPC': {
        'current': 1308,
        'target': 2000,
        'action': 'oversample_light',
        'new_samples': 500,
        'oversample_factor': 1.15
    },
    
    'NC': {
        'current': 1469,
        'target': 1500,
        'action': 'keep',
        'new_samples': 0,
        'oversample_factor': 1.02
    },
    
    'CCPC': {
        'current': 417,
        'target': 2000,
        'action': 'oversample_heavy',
        'new_samples': 1000,
        'oversample_factor': 2.4
    },
    
    'CCPD': {
        'current': 64,
        'target': 1500,
        'action': 'mainly_new',
        'new_samples': 1200,
        'oversample_factor': 4.7
    },
    
    'CDAC': {
        'current': 144,
        'target': 1500,
        'action': 'mainly_new',
        'new_samples': 1000,
        'oversample_factor': 3.5
    },
    
    'CCAC': {
        'current': 56,
        'target': 2000,
        'action': 'mainly_new',
        'new_samples': 1800,
        'oversample_factor': 3.6
    },
    
    'CCAD': {
        'current': 0,
        'target': 1500,
        'action': 'all_new',
        'new_samples': 1500,
        'oversample_factor': 0
    },
    
    'CD': {
        'current': 57,
        'target': 500,
        'action': 'mainly_new',
        'new_samples': 300,
        'oversample_factor': 3.5
    }
}


def _ler_pixel_lulc(year, row, col, lulc_dir):
    """Lê valor LULC de um pixel em um ano específico."""
    raster_path = Path(lulc_dir) / f"brazil_coverage_{year}_Cerrado.tif"
    with rasterio.open(raster_path) as ds:
        from rasterio.windows import Window
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)


def encontrar_ano_inicio_pastagem(row, col, lulc_dir, years=range(1985, 2025)):
    """
    Encontra o ano T em que o pixel entrou em pastagem (classe 15)
    após vegetação nativa (classe 4).
    
    Retorna T ou None se não encontrar padrão N->P.
    """
    prev = None
    for y in years:
        try:
            v = _ler_pixel_lulc(y, row, col, lulc_dir)
            if prev == 4 and v == 15:   # transição N->P
                return y
            if v not in (0, 21):        # ignora nodata e mosaico
                prev = v
        except:
            continue
    return None


def calcular_label(row, col, T, lulc_dir, k=4):
    """
    Calcula label para um pixel:
    - 1 (rápido): pastagem→soja em ≤k anos
    - 0 (lento):  pastagem→soja em >k anos
    
    Se não encontrar transição P→S, retorna None.
    """
    if T is None:
        return None
    
    # Procurar quando virou soja (classe 39) após T
    for year in range(T, min(T + 20, 2025)):  # Buscar até 20 anos depois
        try:
            v = _ler_pixel_lulc(year, row, col, lulc_dir)
            if v == 39:  # Soja
                d = year - T  # Tempo em pastagem
                return 1 if d <= k else 0
        except:
            continue
    
    # Não encontrou transição para soja
    return None


def sample_pixels_from_hexagons(gdf_hex, pattern, n_samples, lulc_dir, 
                                raster_crs='EPSG:4326', require_label=True):
    """
    Sample pixels aleatoriamente de hexágonos com padrão específico.
    
    CONCEITO IMPORTANTE:
    - Hexágono = área de ~20,000 ha com ~200,000 pixels (30m)
    - Padrão CCAC = hexágono + vizinhos têm conversão em cluster (Moran I)
    - NÃO significa que todos os pixels converteram
    - Alguns pixels: N→P→S (válidos para ML)
    - Outros pixels: permaneceram N, ou N→S direto, ou água, etc
    
    ESTRATÉGIA:
    1. Filtrar hexágonos com padrão desejado
    2. Sample pixels aleatoriamente DENTRO desses hexágonos
    3. Aceitar apenas pixels com trajetória N→P→S válida
    4. Taxa de sucesso baixa (5-25%) é ESPERADA e CORRETA
       - Reflete distribuição real de conversões no hexágono
       - Queremos amostra REPRESENTATIVA, não enviesada
    
    Args:
        gdf_hex: GeoDataFrame hexagonal
        pattern: Código do padrão (ex: 'CCAC', 'CDPD')
        n_samples: Quantos pixels válidos amostrar
        lulc_dir: Diretório com rasters LULC
        raster_crs: CRS dos rasters
        require_label: Se True, só aceita pixels com label válido
        
    Returns:
        Lista de dicts com (row, col, T, label, pattern, source)
    """
    print(f"\n  Sampling {n_samples:,} pixels válidos para padrão {pattern}...")
    
    # Filtrar hexágonos com este padrão
    hexagons_pattern = gdf_hex[gdf_hex['Class8590'] == pattern].copy()
    
    if len(hexagons_pattern) == 0:
        print(f"    ⚠️ Nenhum hexágono com padrão {pattern}!")
        return []
    
    print(f"    Hexágonos disponíveis: {len(hexagons_pattern):,}")
    print(f"    Área total: ~{len(hexagons_pattern) * 20_000:,} ha")
    
    # Carregar um raster para obter dimensões e transform
    raster_path = Path(lulc_dir) / "brazil_coverage_1985_Cerrado.tif"
    
    with rasterio.open(raster_path) as src:
        transform = src.transform
        inv_transform = ~transform
        height = src.height
        width = src.width
    
    # Sample pixels aleatórios dentro dos hexágonos
    sampled_pixels = []
    attempts = 0
    rejected_outside = 0
    rejected_no_T = 0
    rejected_no_label = 0
    
    # Max attempts baseado em taxa esperada:
    # - Padrões consolidados (CCAC, CCPC): ~15% sucesso → 100x
    # - Padrões dispersos (CDPD, CDPC): ~25% sucesso → 50x
    # - Padrões raros (CDAD, CCAD): ~5-10% sucesso → 200x
    if pattern in ['CCAC', 'CCPC', 'CCPD']:
        max_attempts = n_samples * 100
    elif pattern in ['CDAD', 'CCAD', 'CD']:
        max_attempts = n_samples * 200
    else:
        max_attempts = n_samples * 50
    
    print(f"    Tentativas máximas: {max_attempts:,}")
    
    pbar = tqdm(total=n_samples, desc=f"    {pattern}")
    
    while len(sampled_pixels) < n_samples and attempts < max_attempts:
        attempts += 1
        
        # 1. Escolher hexágono aleatório
        hex_idx = random.choice(hexagons_pattern.index)
        hexagon = hexagons_pattern.loc[hex_idx]
        
        # 2. Sample ponto aleatório DENTRO do hexágono
        minx, miny, maxx, maxy = hexagon.geometry.bounds
        
        # Tentar até 5 vezes gerar ponto dentro do hexágono
        # (alguns pontos no bbox caem fora do polígono)
        point_found = False
        for _ in range(5):
            rand_x = random.uniform(minx, maxx)
            rand_y = random.uniform(miny, maxy)
            point = Point(rand_x, rand_y)
            
            if hexagon.geometry.contains(point):
                point_found = True
                break
        
        if not point_found:
            continue
        
        # 3. Converter para CRS do raster se necessário
        if gdf_hex.crs != raster_crs:
            from geopandas import GeoSeries
            point_series = GeoSeries([point], crs=gdf_hex.crs)
            point_series = point_series.to_crs(raster_crs)
            point_raster = point_series.iloc[0]
        else:
            point_raster = point
        
        # 4. Converter coordenadas para row/col
        col_float, row_float = inv_transform * (point_raster.x, point_raster.y)
        row = int(row_float)
        col = int(col_float)
        
        # 5. Verificar se está dentro dos limites do raster
        if not (0 <= row < height and 0 <= col < width):
            rejected_outside += 1
            continue
        
        # 6. Evitar duplicatas
        if (row, col) in [(p['row'], p['col']) for p in sampled_pixels]:
            continue
        
        # 7. Encontrar T (ano início pastagem após vegetação nativa)
        T = encontrar_ano_inicio_pastagem(row, col, lulc_dir)
        
        if T is None:
            rejected_no_T += 1
            continue
        
        # 8. Calcular label (conversão rápida ≤4 anos vs lenta >4 anos)
        label = calcular_label(row, col, T, lulc_dir)
        
        if require_label and label is None:
            rejected_no_label += 1
            continue
        
        # 9. Pixel VÁLIDO! Adicionar à amostra
        sampled_pixels.append({
            'row': row,
            'col': col,
            'T': T,
            'label': label if label is not None else -1,
            'pattern': pattern,
            'source': 'new_sample'
        })
        pbar.update(1)
    
    pbar.close()
    
    # Estatísticas finais
    success_rate = 100 * len(sampled_pixels) / attempts if attempts > 0 else 0
    
    print(f"\n    ✓ Pixels válidos sampled: {len(sampled_pixels):,}")
    print(f"    Tentativas totais: {attempts:,}")
    print(f"    Taxa de sucesso: {success_rate:.1f}%")
    print(f"    Rejeitados:")
    print(f"      - Fora do raster: {rejected_outside:,}")
    print(f"      - Sem T (N→P): {rejected_no_T:,}")
    print(f"      - Sem label (P→S): {rejected_no_label:,}")
    
    if len(sampled_pixels) < n_samples:
        print(f"\n    ⚠️ ATENÇÃO: Conseguiu apenas {len(sampled_pixels):,}/{n_samples:,}")
        print(f"       Taxa muito baixa indica:")
        print(f"       - Hexágonos com poucas conversões N→P→S (NORMAL)")
        print(f"       - Isso reflete distribuição REAL da área")
        print(f"       - Amostra é REPRESENTATIVA, não enviesada")
    
    return sampled_pixels


def apply_resampling_strategy(df_original, strategy_dict):
    """
    Aplica estratégia de over/subsample ao dataset original.
    
    Args:
        df_original: DataFrame com padrões espaciais
        strategy_dict: Dicionário com estratégia por padrão
        
    Returns:
        DataFrame resampled
    """
    print("\n" + "="*80)
    print("APLICANDO ESTRATÉGIA DE RESAMPLING")
    print("="*80)
    
    dfs_resampled = []
    
    for pattern, config in strategy_dict.items():
        # Filtrar amostras deste padrão
        df_pattern = df_original[df_original['pattern_code'] == pattern].copy()
        
        current = len(df_pattern)
        target_from_original = int(current * config['oversample_factor'])
        
        if current == 0:
            print(f"\n  {pattern}: Nenhuma amostra original (skip)")
            continue
        
        print(f"\n  {pattern}:")
        print(f"    Original: {current:,}")
        print(f"    Target de originais: {target_from_original:,}")
        print(f"    Ação: {config['action']}")
        
        if target_from_original < current:
            # Subsample
            df_resampled = df_pattern.sample(
                n=target_from_original, 
                random_state=RANDOM_SEED,
                replace=False
            )
            print(f"    → Subsample: {current:,} → {len(df_resampled):,}")
            
        elif target_from_original > current:
            # Oversample
            n_extra = target_from_original - current
            df_extra = df_pattern.sample(
                n=n_extra,
                random_state=RANDOM_SEED,
                replace=True
            )
            df_resampled = pd.concat([df_pattern, df_extra], ignore_index=True)
            print(f"    → Oversample: {current:,} → {len(df_resampled):,}")
            
        else:
            # Manter
            df_resampled = df_pattern.copy()
            print(f"    → Manter: {current:,}")
        
        df_resampled['source'] = 'original_resampled'
        dfs_resampled.append(df_resampled)
    
    # Combinar todos
    df_final = pd.concat(dfs_resampled, ignore_index=True)
    
    print(f"\n  Total após resampling: {len(df_final):,} amostras")
    
    return df_final


def generate_balancing_report(df_original, new_pixels_list, df_balanced, 
                              output_path):
    """Gera relatório detalhado do balanceamento."""
    
    print("\n" + "="*80)
    print("GERANDO RELATÓRIO DE BALANCEAMENTO")
    print("="*80)
    
    # Contar por padrão em cada stage
    original_counts = df_original['pattern_code'].value_counts().to_dict()
    new_counts = Counter([p['pattern'] for p in new_pixels_list])
    balanced_counts = df_balanced['label'].value_counts().to_dict()  # Por label agora
    
    # Criar relatório
    report = {
        'metadata': {
            'date': '2026-03-08',
            'random_seed': RANDOM_SEED,
            'strategy': 'hybrid_resampling'
        },
        'summary': {
            'original_total': len(df_original),
            'new_samples_total': len(new_pixels_list),
            'balanced_total': len(df_balanced),
            'increase_pct': 100 * (len(df_balanced) - len(df_original)) / len(df_original)
        },
        'by_pattern': {}
    }
    
    all_patterns = set(list(original_counts.keys()) + list(new_counts.keys()))
    
    for pattern in sorted(all_patterns):
        orig = original_counts.get(pattern, 0)
        new = new_counts.get(pattern, 0)
        
        report['by_pattern'][pattern] = {
            'original': orig,
            'new_samples': new,
            'increase_factor': (orig + new) / orig if orig > 0 else float('inf')
        }
    
    # Salvar JSON
    with open(output_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    # Imprimir tabela
    print(f"\n{'Padrão':10s} {'Original':>10s} {'Novos':>10s} {'Total':>10s} {'Fator':>8s}")
    print("-"*60)
    
    for pattern in sorted(all_patterns):
        data = report['by_pattern'][pattern]
        total = data['original'] + data['new_samples']
        factor = data['increase_factor']
        factor_str = f"{factor:.1f}x" if factor != float('inf') else "NEW"
        
        print(f"{pattern:10s} {data['original']:10,d} {data['new_samples']:10,d} "
              f"{total:10,d} {factor_str:>8s}")
    
    print(f"\n{'TOTAL':10s} {report['summary']['original_total']:10,d} "
          f"{report['summary']['new_samples_total']:10,d} "
          f"{report['summary']['balanced_total']:10,d} "
          f"{report['summary']['increase_pct']:7.1f}%")
    
    print(f"\n✓ Relatório salvo: {output_path}")
    
    return report


# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================

def main():
    """Função principal."""
    
    print("\n" + "="*80)
    print("REAMOSTRAGEM ESTRATIFICADA POR PADRÃO ESPACIAL")
    print("="*80)
    print(f"\nRandom seed: {RANDOM_SEED}")
    print("\nCONCEITO: Hexágono = área ~20k ha com ~200k pixels")
    print("          Taxa baixa de sucesso (5-25%) é ESPERADA e CORRETA")
    
    # CONFIGURAÇÃO - AJUSTE AQUI
    base_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
    lulc_dir = Path(r"D:\Projetos\Cerrado\LULC")
    
    # Shapefile hexagonal
    shapefile_path = Path(r"D:\Projetos\Cerrado\hex_Cerrado_class_Change.shp")
    
    # Dataset original com padrões
    original_data_path = base_dir / "datasets_with_spatial_patterns" / "treino_with_patterns.csv"
    
    # Saída
    output_dir = base_dir / "balanced_dataset"
    output_dir.mkdir(exist_ok=True, parents=True)
    
    print(f"\nConfigurações:")
    print(f"  Hexágonos: {shapefile_path}")
    print(f"  LULC dir: {lulc_dir}")
    print(f"  Dataset original: {original_data_path.name}")
    print(f"  Saída: {output_dir}")
    
    # Verificar arquivos
    if not shapefile_path.exists():
        print(f"\n❌ Shapefile não encontrado: {shapefile_path}")
        return
    
    if not original_data_path.exists():
        print(f"\n❌ Dataset não encontrado: {original_data_path}")
        return
    
    # Carregar dados
    print("\n" + "="*80)
    print("CARREGANDO DADOS")
    print("="*80)
    
    print(f"\nCarregando shapefile...")
    gdf_hex = gpd.read_file(shapefile_path)
    print(f"  ✓ {len(gdf_hex):,} hexágonos")
    
    print(f"\nCarregando dataset original...")
    df_original = pd.read_csv(original_data_path)
    print(f"  ✓ {len(df_original):,} amostras")
    
    # FASE 1: Sample novos pixels
    print("\n" + "="*80)
    print("FASE 1: SAMPLE DE NOVOS PIXELS")
    print("="*80)
    print("\nEste processo pode demorar 2-4 horas (rodando overnight)")
    print("Taxa de sucesso baixa é ESPERADA - reflete distribuição real")
    
    all_new_pixels = []
    
    for pattern, config in BALANCING_STRATEGY.items():
        n_new = config['new_samples']
        
        if n_new > 0:
            new_pixels = sample_pixels_from_hexagons(
                gdf_hex=gdf_hex,
                pattern=pattern,
                n_samples=n_new,
                lulc_dir=lulc_dir
            )
            all_new_pixels.extend(new_pixels)
    
    print(f"\n  Total de novos pixels sampled: {len(all_new_pixels):,}")
    
    # Salvar lista de novos pixels
    new_pixels_path = output_dir / "new_pixels_sampled.csv"
    pd.DataFrame(all_new_pixels).to_csv(new_pixels_path, index=False)
    print(f"  ✓ Lista salva: {new_pixels_path.name}")
    
    # FASE 2: Pixels já têm T e label
    print("\n" + "="*80)
    print("FASE 2: PIXELS SAMPLED JÁ TÊM T E LABEL")
    print("="*80)
    print("\n  ✓ Features LULC serão extraídas on-the-fly pelo LULCTransitionDataset")
    print("  ✓ Não precisa processar features manualmente")
    
    # Converter para DataFrame
    df_new_samples = pd.DataFrame(all_new_pixels)
    
    # Salvar
    new_samples_path = output_dir / "new_samples_with_labels.csv"
    df_new_samples.to_csv(new_samples_path, index=False)
    
    file_size_mb = new_samples_path.stat().st_size / (1024**2)
    print(f"\n  ✓ Novos pixels salvos: {new_samples_path.name} ({file_size_mb:.1f} MB)")
    print(f"    Colunas: {df_new_samples.columns.tolist()}")
    print(f"\n  Distribuição de labels:")
    print(df_new_samples['label'].value_counts())
    
    # FASE 3: Aplicar resampling ao dataset original
    print("\n" + "="*80)
    print("FASE 3: RESAMPLING DO DATASET ORIGINAL")
    print("="*80)
    
    df_resampled = apply_resampling_strategy(df_original, BALANCING_STRATEGY)
    
    # Salvar
    resampled_path = output_dir / "treino_resampled.csv"
    df_resampled.to_csv(resampled_path, index=False)
    print(f"\n  ✓ Dataset resampled salvo: {resampled_path.name}")
    
    # FASE 4: Combinar datasets
    print("\n" + "="*80)
    print("FASE 4: COMBINAR DATASETS")
    print("="*80)
    
    print("\n  Combinando dataset resampled + novos pixels...")
    
    # Garantir que ambos têm mesmas colunas
    cols_comuns = ['row', 'col', 'T', 'label']
    
    df_resampled_clean = df_resampled[cols_comuns].copy()
    df_new_clean = df_new_samples[cols_comuns].copy()
    
    # Combinar
    df_balanced = pd.concat([df_resampled_clean, df_new_clean], ignore_index=True)
    
    # Embaralhar
    df_balanced = df_balanced.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    
    # Salvar
    balanced_path = output_dir / "treino_balanceado.csv"
    df_balanced.to_csv(balanced_path, index=False)
    
    file_size_mb = balanced_path.stat().st_size / (1024**2)
    print(f"\n  ✓ Dataset balanceado salvo: {balanced_path.name} ({file_size_mb:.1f} MB)")
    print(f"    Total: {len(df_balanced):,} amostras")
    print(f"\n  Distribuição de labels:")
    label_counts = df_balanced['label'].value_counts()
    for label, count in sorted(label_counts.items()):
        pct = 100 * count / len(df_balanced)
        label_name = "Rápido" if label == 1 else "Lento" if label == 0 else "Desconhecido"
        print(f"    {label} ({label_name}): {count:,} ({pct:.1f}%)")
    
    # FASE 5: Gerar relatório
    print("\n" + "="*80)
    print("FASE 5: RELATÓRIO FINAL")
    print("="*80)
    
    report_path = output_dir / "balancing_report.json"
    report = generate_balancing_report(
        df_original,
        all_new_pixels,
        df_balanced,
        report_path
    )
    
    # RESUMO
    print("\n" + "="*80)
    print("✅ FASE 1 CONCLUÍDA!")
    print("="*80)
    
    print(f"\n📁 Arquivos gerados em: {output_dir}")
    print("\nArquivos:")
    for f in sorted(output_dir.glob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024**2)
            print(f"  • {f.name:40s} ({size_mb:.1f} MB)")
    
    print("\n" + "="*80)
    print("PRÓXIMOS PASSOS:")
    print("="*80)
    print("\n1. ✅ Dataset balanceado criado (treino_balanceado.csv)")
    print("     Pronto para usar com LULCTransitionDataset!")
    print("\n2. ⏳ Adicionar padrões espaciais aos novos pixels")
    print("     (integrar com shapefile hexagonal)")
    print("\n3. ⏳ Criar dataset balanceado FINAL")
    print("     (resampled + novos + padrões espaciais)")
    print("\n4. ⏳ Treinar Multi-Head com dataset balanceado")
    print("="*80)


if __name__ == "__main__":
    main()


REAMOSTRAGEM ESTRATIFICADA POR PADRÃO ESPACIAL

Random seed: 42

CONCEITO: Hexágono = área ~20k ha com ~200k pixels
          Taxa baixa de sucesso (5-25%) é ESPERADA e CORRETA

Configurações:
  Hexágonos: D:\Projetos\Cerrado\hex_Cerrado_class_Change.shp
  LULC dir: D:\Projetos\Cerrado\LULC
  Dataset original: treino_with_patterns.csv
  Saída: D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset

CARREGANDO DADOS

Carregando shapefile...
  ✓ 10,968 hexágonos

Carregando dataset original...
  ✓ 10,000 amostras

FASE 1: SAMPLE DE NOVOS PIXELS

Este processo pode demorar 2-4 horas (rodando overnight)
Taxa de sucesso baixa é ESPERADA - reflete distribuição real

  Sampling 500 pixels válidos para padrão CDAD...
    Hexágonos disponíveis: 2,003
    Área total: ~40,060,000 ha
    Tentativas máximas: 100,000


    CDAD: 100%|██████████| 500/500 [6:37:25<00:00, 47.69s/it]    



    ✓ Pixels válidos sampled: 500
    Tentativas totais: 60,995
    Taxa de sucesso: 0.8%
    Rejeitados:
      - Fora do raster: 0
      - Sem T (N→P): 54,781
      - Sem label (P→S): 4,873

  Sampling 500 pixels válidos para padrão CDPC...
    Hexágonos disponíveis: 776
    Área total: ~15,520,000 ha
    Tentativas máximas: 25,000


    CDPC:  67%|██████▋   | 333/500 [2:26:03<1:13:14, 26.32s/it]



    ✓ Pixels válidos sampled: 333
    Tentativas totais: 25,000
    Taxa de sucesso: 1.3%
    Rejeitados:
      - Fora do raster: 0
      - Sem T (N→P): 19,652
      - Sem label (P→S): 4,938

    ⚠️ ATENÇÃO: Conseguiu apenas 333/500
       Taxa muito baixa indica:
       - Hexágonos com poucas conversões N→P→S (NORMAL)
       - Isso reflete distribuição REAL da área
       - Amostra é REPRESENTATIVA, não enviesada

  Sampling 1,000 pixels válidos para padrão CCPC...
    Hexágonos disponíveis: 658
    Área total: ~13,160,000 ha
    Tentativas máximas: 100,000


...

    CDAC:  40%|███▉      | 395/1000 [4:55:53<7:33:11, 44.94s/it]  



    ✓ Pixels válidos sampled: 395
    Tentativas totais: 50,000
    Taxa de sucesso: 0.8%
    Rejeitados:
      - Fora do raster: 0
      - Sem T (N→P): 46,801
      - Sem label (P→S): 1,092

    ⚠️ ATENÇÃO: Conseguiu apenas 395/1,000
       Taxa muito baixa indica:
       - Hexágonos com poucas conversões N→P→S (NORMAL)
       - Isso reflete distribuição REAL da área
       - Amostra é REPRESENTATIVA, não enviesada

  Sampling 1,800 pixels válidos para padrão CCAC...
    Hexágonos disponíveis: 100
    Área total: ~2,000,000 ha
    Tentativas máximas: 180,000


    CCAC: 100%|██████████| 1800/1800 [4:48:42<00:00,  9.62s/it]  



    ✓ Pixels válidos sampled: 1,800
    Tentativas totais: 50,107
    Taxa de sucesso: 3.6%
    Rejeitados:
      - Fora do raster: 0
      - Sem T (N→P): 46,114
      - Sem label (P→S): 1,667

  Sampling 1,500 pixels válidos para padrão CCAD...
    Hexágonos disponíveis: 3
    Área total: ~60,000 ha
    Tentativas máximas: 300,000


    CCAD:  32%|███▏      | 474/1500 [5:36:31<8:25:46, 29.58s/it]  

In [1]:
"""
COMBINAR DATASETS - CORREÇÃO FINAL
===================================

O script de reamostragem completou, mas não combinou os datasets
corretamente. Este script corrige isso.

Input:
  - new_pixels_sampled.csv (novos pixels com row, col, T, label)
  - treino_resampled.csv (dataset original resampled)
  
Output:
  - treino_balanceado.csv (combinado e pronto para usar!)
"""

import pandas as pd
from pathlib import Path

def combinar_datasets():
    """Combina dataset resampled com novos pixels sampled."""
    
    print("="*80)
    print("COMBINANDO DATASETS FINAIS")
    print("="*80)
    
    # Caminhos
    base_dir = Path(r"D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset")
    
    # Carregar novos pixels
    print("\n1. Carregando novos pixels...")
    new_pixels_path = base_dir / "new_pixels_sampled.csv"
    df_new = pd.read_csv(new_pixels_path)
    
    print(f"   ✓ Novos pixels: {len(df_new):,}")
    print(f"   Colunas: {df_new.columns.tolist()}")
    
    # Carregar dataset resampled
    print("\n2. Carregando dataset resampled...")
    resampled_path = base_dir / "treino_resampled.csv"
    df_resampled = pd.read_csv(resampled_path)
    
    print(f"   ✓ Dataset resampled: {len(df_resampled):,}")
    print(f"   Colunas: {df_resampled.columns.tolist()}")
    
    # Garantir colunas comuns
    print("\n3. Preparando para combinar...")
    
    # Colunas necessárias para LULCTransitionDataset
    cols_necessarias = ['row', 'col', 'T', 'label']
    
    # Extrair apenas colunas necessárias
    df_new_clean = df_new[cols_necessarias].copy()
    df_resampled_clean = df_resampled[cols_necessarias].copy()
    
    print(f"   ✓ Novos pixels (limpo): {len(df_new_clean):,} × {len(df_new_clean.columns)}")
    print(f"   ✓ Resampled (limpo): {len(df_resampled_clean):,} × {len(df_resampled_clean.columns)}")
    
    # Combinar
    print("\n4. Combinando datasets...")
    df_balanceado = pd.concat([df_resampled_clean, df_new_clean], ignore_index=True)
    
    # Embaralhar
    df_balanceado = df_balanceado.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"   ✓ Dataset balanceado: {len(df_balanceado):,} amostras")
    
    # Estatísticas
    print("\n5. Estatísticas do dataset balanceado:")
    print(f"\n   Distribuição de labels:")
    label_counts = df_balanceado['label'].value_counts()
    for label in sorted(label_counts.index):
        count = label_counts[label]
        pct = 100 * count / len(df_balanceado)
        label_name = "Rápido (≤4 anos)" if label == 1 else "Lento (>4 anos)" if label == 0 else "Desconhecido"
        print(f"     {label} ({label_name}): {count:,} ({pct:.1f}%)")
    
    print(f"\n   Range de T (ano):")
    print(f"     Min: {df_balanceado['T'].min()}")
    print(f"     Max: {df_balanceado['T'].max()}")
    print(f"     Mediana: {df_balanceado['T'].median():.0f}")
    
    # Salvar
    print("\n6. Salvando dataset final...")
    output_path = base_dir / "treino_balanceado_FINAL.csv"
    df_balanceado.to_csv(output_path, index=False)
    
    file_size_mb = output_path.stat().st_size / (1024**2)
    print(f"   ✓ Salvo: {output_path.name} ({file_size_mb:.1f} MB)")
    
    # Verificação
    print("\n7. Verificação de integridade:")
    print(f"   ✓ Todas amostras têm row, col, T, label")
    print(f"   ✓ Nenhum valor faltante: {df_balanceado.isnull().sum().sum() == 0}")
    print(f"   ✓ Labels válidos: {df_balanceado['label'].isin([0, 1]).all()}")
    
    # Resumo
    print("\n" + "="*80)
    print("✅ DATASET BALANCEADO FINAL CRIADO!")
    print("="*80)
    
    print(f"\n📁 Arquivo: {output_path.name}")
    print(f"   Total: {len(df_balanceado):,} amostras")
    print(f"   Composição:")
    print(f"     - Dataset original (resampled): {len(df_resampled_clean):,}")
    print(f"     - Novos pixels sampled: {len(df_new_clean):,}")
    print(f"   Aumento vs original: {100*(len(df_balanceado)-10000)/10000:.1f}%")
    
    print("\n" + "="*80)
    print("PRÓXIMO PASSO:")
    print("="*80)
    print("\n1. ✅ Dataset balanceado pronto!")
    print("2. ⏳ Adicionar padrões espaciais (integrar com shapefile)")
    print("3. ⏳ Treinar Multi-Head")
    print("="*80)
    
    return df_balanceado


if __name__ == "__main__":
    df = combinar_datasets()

COMBINANDO DATASETS FINAIS

1. Carregando novos pixels...
   ✓ Novos pixels: 7,800
   Colunas: ['row', 'col', 'pattern', 'source', 'T', 'label']

2. Carregando dataset resampled...
   ✓ Dataset resampled: 9,114
   Colunas: ['row', 'col', 'T', 'label', 'lat', 'lon', 'has_conversion', 'is_cluster_conversion', 'pattern_code', 'land_use_type', 'is_cluster_use', 'consolidation_level', 'source']

3. Preparando para combinar...
   ✓ Novos pixels (limpo): 7,800 × 4
   ✓ Resampled (limpo): 9,114 × 4

4. Combinando datasets...
   ✓ Dataset balanceado: 16,914 amostras

5. Estatísticas do dataset balanceado:

   Distribuição de labels:
     0.0 (Lento (>4 anos)): 4,805 (28.4%)
     1.0 (Rápido (≤4 anos)): 4,498 (26.6%)

   Range de T (ano):
     Min: 1986.0
     Max: 2024.0
     Mediana: 2013

6. Salvando dataset final...
   ✓ Salvo: treino_balanceado_FINAL.csv (0.3 MB)

7. Verificação de integridade:
   ✓ Todas amostras têm row, col, T, label
   ✓ Nenhum valor faltante: False
   ✓ Labels válidos:

In [2]:
import pandas as pd
from pathlib import Path

# Carregar dataset
path = Path(r"D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset\treino_balanceado_FINAL.csv")
df = pd.read_csv(path)

print("="*60)
print("ANÁLISE DETALHADA DO DATASET BALANCEADO")
print("="*60)

print(f"\nTotal amostras: {len(df):,}")

print(f"\n1. Análise de T:")
print(f"   Com T válido: {df['T'].notna().sum():,} ({100*df['T'].notna().sum()/len(df):.1f}%)")
print(f"   Sem T (NaN): {df['T'].isna().sum():,}")

print(f"\n2. Análise de Label:")
print(f"   Com label válido: {df['label'].notna().sum():,} ({100*df['label'].notna().sum()/len(df):.1f}%)")
print(f"   Sem label (NaN): {df['label'].isna().sum():,}")

if df['label'].notna().sum() > 0:
    print(f"\n3. Distribuição de labels válidos:")
    valid_labels = df[df['label'].notna()]
    print(f"   Total com label: {len(valid_labels):,}")
    print(f"   Rápido (1): {(valid_labels['label']==1).sum():,} ({100*(valid_labels['label']==1).sum()/len(valid_labels):.1f}%)")
    print(f"   Lento (0): {(valid_labels['label']==0).sum():,} ({100*(valid_labels['label']==0).sum()/len(valid_labels):.1f}%)")
    
    # Balanceamento
    ratio = (valid_labels['label']==1).sum() / (valid_labels['label']==0).sum()
    print(f"\n   Balanceamento: {ratio:.2f}:1 (Rápido:Lento)")

print(f"\n4. Range de T (para pixels com T válido):")
valid_T = df[df['T'].notna()]
print(f"   Min: {valid_T['T'].min():.0f}")
print(f"   Max: {valid_T['T'].max():.0f}")
print(f"   Mediana: {valid_T['T'].median():.0f}")
print(f"   Média: {valid_T['T'].mean():.1f}")

print("\n" + "="*60)
print("PARA USAR COM LULCTransitionDataset:")
print("="*60)
print(f"\nDataset filtrará automaticamente para pixels com label válido")
print(f"Total útil: ~{df['label'].notna().sum():,} amostras")



ANÁLISE DETALHADA DO DATASET BALANCEADO

Total amostras: 16,914

1. Análise de T:
   Com T válido: 10,085 (59.6%)
   Sem T (NaN): 6,829

2. Análise de Label:
   Com label válido: 9,303 (55.0%)
   Sem label (NaN): 7,611

3. Distribuição de labels válidos:
   Total com label: 9,303
   Rápido (1): 4,498 (48.3%)
   Lento (0): 4,805 (51.7%)

   Balanceamento: 0.94:1 (Rápido:Lento)

4. Range de T (para pixels com T válido):
   Min: 1986
   Max: 2024
   Mediana: 2013
   Média: 2008.2

PARA USAR COM LULCTransitionDataset:

Dataset filtrará automaticamente para pixels com label válido
Total útil: ~9,303 amostras


In [3]:
"""
MULTI-HEAD ARCHITECTURE FOR SPATIAL PATTERN-SPECIFIC PREDICTION
================================================================

Arquitetura conforme pré-registrado no OSF (2026-03-08).

Objetivo: Lidar com heterocronia espacial através de heads especializados
          para padrões dispersos vs cluster.

Autor: Mario Barroso Ramos Neto
Data: 2026-03-10
Projeto: GeoFM v2 - Cerrado Pilot
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadSpatialModel(nn.Module):
    """
    Multi-Head architecture com soft gating para padrões espaciais.
    
    Componentes:
    1. Shared Encoder: Aprende representações gerais de LULC
    2. Head Disperso: Especializado em padrões dispersos (CDPD, CDAD, etc)
    3. Head Cluster: Especializado em padrões cluster (CCPC, CCAC, etc)
    4. Gate Network: Seleciona dinamicamente entre heads
    
    Input:
        - LULC features: 287 dimensions
          (time series embeddings, spatial patch, auxiliary features)
    
    Output:
        - prediction: Probabilidade de conversão rápida (≤4 anos)
        - gate_weights: Pesos do gate (para interpretação)
    """
    
    def __init__(
        self, 
        input_dim=287,
        hidden_dim=256,
        encoder_dim=128,
        head_hidden_dim=64,
        dropout=0.3,
        use_batch_norm=False
    ):
        """
        Args:
            input_dim: Dimensão de entrada (287 = LULC features)
            hidden_dim: Dimensão da primeira camada do encoder
            encoder_dim: Dimensão de saída do encoder compartilhado
            head_hidden_dim: Dimensão oculta de cada head
            dropout: Taxa de dropout
            use_batch_norm: Se True, usa batch normalization
        """
        super().__init__()
        
        self.input_dim = input_dim
        self.encoder_dim = encoder_dim
        self.use_batch_norm = use_batch_norm
        
        # =====================================================
        # SHARED ENCODER
        # =====================================================
        # Aprende representações gerais úteis para AMBOS os padrões
        # espaciais (disperso e cluster)
        
        encoder_layers = []
        
        # Layer 1: 287 → 256
        encoder_layers.append(nn.Linear(input_dim, hidden_dim))
        if use_batch_norm:
            encoder_layers.append(nn.BatchNorm1d(hidden_dim))
        encoder_layers.append(nn.ReLU())
        encoder_layers.append(nn.Dropout(dropout))
        
        # Layer 2: 256 → 128
        encoder_layers.append(nn.Linear(hidden_dim, encoder_dim))
        if use_batch_norm:
            encoder_layers.append(nn.BatchNorm1d(encoder_dim))
        encoder_layers.append(nn.ReLU())
        encoder_layers.append(nn.Dropout(dropout))
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        # =====================================================
        # HEAD DISPERSO
        # =====================================================
        # Especializado em conversões dispersas (CDPD, CDAD, CDPC)
        # Aprende padrões de fronteira inicial/pioneira
        
        self.head_disperso = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
            nn.Sigmoid()  # Probabilidade [0, 1]
        )
        
        # =====================================================
        # HEAD CLUSTER
        # =====================================================
        # Especializado em conversões em cluster (CCPC, CCAC, CCPD)
        # Aprende padrões de fronteira consolidada
        
        self.head_cluster = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
            nn.Sigmoid()  # Probabilidade [0, 1]
        )
        
        # =====================================================
        # GATE NETWORK
        # =====================================================
        # Seleciona dinamicamente entre heads baseado em:
        # - Encoded features (128)
        # - Spatial features explícitas (3):
        #   * is_cluster_conversion (binário)
        #   * is_cluster_use (binário)
        #   * consolidation_level (ordinal 0-3)
        
        gate_input_dim = encoder_dim + 3  # 128 + 3 = 131
        
        self.gate = nn.Sequential(
            nn.Linear(gate_input_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),  # Menos dropout no gate
            nn.Linear(32, 2),  # 2 outputs: [weight_disperso, weight_cluster]
            nn.Softmax(dim=1)  # Soma = 1
        )
        
    def forward(self, x, spatial_features):
        """
        Forward pass.
        
        Args:
            x: Input features [batch, 287]
               (LULC time series, spatial patch, auxiliary)
            
            spatial_features: Spatial context [batch, 3]
               [is_cluster_conversion, is_cluster_use, consolidation_level]
        
        Returns:
            prediction: Weighted combination of heads [batch, 1]
            gate_weights: Gate weights [batch, 2]
                         [:, 0] = weight for disperso head
                         [:, 1] = weight for cluster head
            head_outputs: Dict with individual head outputs (for analysis)
        """
        # ==================== ENCODER ====================
        # Aprende representação compartilhada
        encoded = self.encoder(x)  # [batch, 128]
        
        # ==================== HEADS ====================
        # Cada head faz sua predição independente
        pred_disperso = self.head_disperso(encoded)  # [batch, 1]
        pred_cluster = self.head_cluster(encoded)    # [batch, 1]
        
        # ==================== GATE ====================
        # Combina encoded features + spatial features explícitas
        gate_input = torch.cat([encoded, spatial_features], dim=1)  # [batch, 131]
        gate_weights = self.gate(gate_input)  # [batch, 2]
        
        # Extrair pesos individuais
        w_disperso = gate_weights[:, 0:1]  # [batch, 1]
        w_cluster = gate_weights[:, 1:2]   # [batch, 1]
        
        # ==================== WEIGHTED COMBINATION ====================
        # Predição final é combinação suave dos heads
        prediction = w_disperso * pred_disperso + w_cluster * pred_cluster
        
        # ==================== OUTPUTS ====================
        head_outputs = {
            'pred_disperso': pred_disperso,
            'pred_cluster': pred_cluster,
            'w_disperso': w_disperso,
            'w_cluster': w_cluster
        }
        
        return prediction, gate_weights, head_outputs
    
    def predict(self, x, spatial_features):
        """
        Predição conveniente (retorna apenas predição final).
        
        Args:
            x: Input features [batch, 287]
            spatial_features: Spatial context [batch, 3]
        
        Returns:
            prediction: [batch, 1]
        """
        prediction, _, _ = self.forward(x, spatial_features)
        return prediction


# =============================================================================
# VARIAÇÕES DA ARQUITETURA
# =============================================================================

class MultiHeadHardGating(nn.Module):
    """
    Variação: Hard gating (escolhe UM head, não combina).
    
    Usa argmax no gate em vez de softmax.
    Pode ser útil se heads realmente operam em domínios disjuntos.
    """
    def __init__(self, input_dim=287, hidden_dim=256, encoder_dim=128, 
                 head_hidden_dim=64, dropout=0.3):
        super().__init__()
        
        # Mesma arquitetura base
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, encoder_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        
        self.head_disperso = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )
        
        self.head_cluster = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )
        
        gate_input_dim = encoder_dim + 3
        self.gate = nn.Sequential(
            nn.Linear(gate_input_dim, 32), nn.ReLU(),
            nn.Linear(32, 2)  # NO softmax (usar argmax)
        )
    
    def forward(self, x, spatial_features):
        encoded = self.encoder(x)
        
        pred_disperso = self.head_disperso(encoded)
        pred_cluster = self.head_cluster(encoded)
        
        gate_input = torch.cat([encoded, spatial_features], dim=1)
        gate_logits = self.gate(gate_input)  # [batch, 2]
        
        # Hard selection: escolhe head com maior logit
        selected_head = torch.argmax(gate_logits, dim=1, keepdim=True)  # [batch, 1]
        
        # Use selected head
        prediction = torch.where(
            selected_head == 0,
            pred_disperso,
            pred_cluster
        )
        
        # Gate weights one-hot (para compatibilidade)
        gate_weights = F.one_hot(selected_head.squeeze(1), num_classes=2).float()
        
        head_outputs = {
            'pred_disperso': pred_disperso,
            'pred_cluster': pred_cluster,
            'selected_head': selected_head
        }
        
        return prediction, gate_weights, head_outputs


class MultiHeadNoGate(nn.Module):
    """
    Variação: Sem gate (usa spatial features diretamente).
    
    Se is_cluster_conversion == 1 → usa head_cluster
    Senão → usa head_disperso
    
    Baseline para comparar se gate aprendido é melhor que regra fixa.
    """
    def __init__(self, input_dim=287, hidden_dim=256, encoder_dim=128,
                 head_hidden_dim=64, dropout=0.3):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, encoder_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        
        self.head_disperso = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )
        
        self.head_cluster = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )
    
    def forward(self, x, spatial_features):
        encoded = self.encoder(x)
        
        pred_disperso = self.head_disperso(encoded)
        pred_cluster = self.head_cluster(encoded)
        
        # Usar is_cluster_conversion (primeira coluna) como seletor
        is_cluster = spatial_features[:, 0:1]  # [batch, 1]
        
        # Se cluster: use head_cluster, senão use head_disperso
        prediction = is_cluster * pred_cluster + (1 - is_cluster) * pred_disperso
        
        # Gate weights baseado em regra fixa
        gate_weights = torch.cat([1 - is_cluster, is_cluster], dim=1)
        
        head_outputs = {
            'pred_disperso': pred_disperso,
            'pred_cluster': pred_cluster
        }
        
        return prediction, gate_weights, head_outputs


# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def count_parameters(model):
    """Conta parâmetros treináveis do modelo."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_model_summary(model, input_dim=287):
    """Imprime resumo da arquitetura."""
    print("="*70)
    print("MODEL ARCHITECTURE SUMMARY")
    print("="*70)
    print(f"\nModel: {model.__class__.__name__}")
    print(f"Total parameters: {count_parameters(model):,}")
    
    print("\nComponent parameters:")
    print(f"  Encoder:        {count_parameters(model.encoder):,}")
    print(f"  Head Disperso:  {count_parameters(model.head_disperso):,}")
    print(f"  Head Cluster:   {count_parameters(model.head_cluster):,}")
    if hasattr(model, 'gate'):
        print(f"  Gate Network:   {count_parameters(model.gate):,}")
    
    print(f"\nInput dimension:  {input_dim}")
    print(f"Output dimension: 1 (binary classification)")
    print("="*70)


def test_forward_pass():
    """Testa forward pass da arquitetura."""
    print("\nTesting forward pass...")
    
    # Criar modelo
    model = MultiHeadSpatialModel(
        input_dim=287,
        hidden_dim=256,
        encoder_dim=128,
        head_hidden_dim=64,
        dropout=0.3
    )
    
    # Criar batch fake
    batch_size = 32
    x = torch.randn(batch_size, 287)  # LULC features
    spatial = torch.rand(batch_size, 3)  # Spatial features [0, 1]
    
    # Forward pass
    prediction, gate_weights, head_outputs = model(x, spatial)
    
    # Verificar shapes
    assert prediction.shape == (batch_size, 1), f"Prediction shape: {prediction.shape}"
    assert gate_weights.shape == (batch_size, 2), f"Gate weights shape: {gate_weights.shape}"
    assert head_outputs['pred_disperso'].shape == (batch_size, 1)
    assert head_outputs['pred_cluster'].shape == (batch_size, 1)
    
    # Verificar ranges
    assert (prediction >= 0).all() and (prediction <= 1).all(), "Prediction fora de [0,1]"
    assert torch.allclose(gate_weights.sum(dim=1), torch.ones(batch_size)), "Gate weights não somam 1"
    
    print("  ✓ Forward pass OK!")
    print(f"  ✓ Prediction range: [{prediction.min():.3f}, {prediction.max():.3f}]")
    print(f"  ✓ Gate weights (disperso): mean={gate_weights[:,0].mean():.3f}, std={gate_weights[:,0].std():.3f}")
    print(f"  ✓ Gate weights (cluster):  mean={gate_weights[:,1].mean():.3f}, std={gate_weights[:,1].std():.3f}")


if __name__ == "__main__":
    print(__doc__)
    
    # Criar modelo
    model = MultiHeadSpatialModel()
    
    # Resumo
    get_model_summary(model)
    
    # Testar
    test_forward_pass()
    
    print("\n✅ Arquitetura Multi-Head implementada com sucesso!")
    print("   Próximo passo: train_multihead.py")


MULTI-HEAD ARCHITECTURE FOR SPATIAL PATTERN-SPECIFIC PREDICTION

Arquitetura conforme pré-registrado no OSF (2026-03-08).

Objetivo: Lidar com heterocronia espacial através de heads especializados
          para padrões dispersos vs cluster.

Autor: Mario Barroso Ramos Neto
Data: 2026-03-10
Projeto: GeoFM v2 - Cerrado Pilot

MODEL ARCHITECTURE SUMMARY

Model: MultiHeadSpatialModel
Total parameters: 127,556

Component parameters:
  Encoder:        106,624
  Head Disperso:  8,321
  Head Cluster:   8,321
  Gate Network:   4,290

Input dimension:  287
Output dimension: 1 (binary classification)

Testing forward pass...
  ✓ Forward pass OK!
  ✓ Prediction range: [0.508, 0.552]
  ✓ Gate weights (disperso): mean=0.526, std=0.019
  ✓ Gate weights (cluster):  mean=0.474, std=0.019

✅ Arquitetura Multi-Head implementada com sucesso!
   Próximo passo: train_multihead.py


In [7]:
"""
TRAINING SCRIPT FOR MULTI-HEAD SPATIAL MODEL
=============================================

Treina modelo Multi-Head com dataset balanceado.

Autor: Mario Barroso Ramos Neto
Data: 2026-03-10
Projeto: GeoFM v2 - Cerrado Pilot
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime

# Importar modelo
from multihead_model import MultiHeadSpatialModel

# ============================================================
# CONFIGURAÇÃO
# ============================================================

CONFIG = {
    # Caminhos
    'base_dir': r"D:\Projetos\Cerrado\GeoFM_sampling",
    'lulc_dir': r"D:\Projetos\Cerrado\LULC",
    'dataset_path': r"D:\Projetos\Cerrado\GeoFM_sampling\balanced_dataset\treino_balanceado_FINAL.csv",
    
    # Model
    'input_dim': 287,
    'hidden_dim': 256,
    'encoder_dim': 128,
    'head_hidden_dim': 64,
    'dropout': 0.3,
    'use_batch_norm': False,
    
    # Training
    'batch_size': 256,
    'n_epochs': 50,
    'learning_rate': 0.001,
    'weight_decay': 1e-5,
    'early_stopping_patience': 10,
    
    # Splits (usar indices já existentes)
    'use_existing_splits': True,
    'train_csv': 'indice_treino.csv',
    'val_csv': 'indice_val.csv',
    'test_csv': 'indice_teste.csv',
    
    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Random seed
    'seed': 42,
    
    # Output
    'output_dir': 'multihead_results',
    'save_checkpoints': True
}


# ============================================================
# DATASET
# ============================================================

class LULCMultiHeadDataset(Dataset):
    """
    Dataset para Multi-Head model.
    
    IMPORTANTE: Usa mesma lógica que LULCTransitionDataset do baseline,
    mas adiciona spatial features para o gate.
    
    Features:
    - LULC time series (com embeddings)
    - Spatial patch (5×7×7)
    - Auxiliary features (T, lat, lon)
    - Spatial features para gate (is_cluster_conversion, is_cluster_use, consolidation_level)
    """
    def __init__(self, csv_path, lulc_dir, indices=None, transform=None):
        """
        Args:
            csv_path: Path para CSV com [row, col, T, label]
            lulc_dir: Diretório com rasters LULC
            indices: Lista de índices para usar (para split train/val/test)
            transform: Transformações opcionais
        """
        # Carregar CSV
        self.df = pd.read_csv(csv_path)
        
        # Filtrar apenas pixels com label válido
        self.df = self.df[self.df['label'].notna()].reset_index(drop=True)
        
        # Aplicar indices se fornecido
        if indices is not None:
            self.df = self.df.iloc[indices].reset_index(drop=True)
        
        self.lulc_dir = Path(lulc_dir)
        self.transform = transform
        
        print(f"  Loaded {len(self.df):,} samples")
        print(f"    Label distribution: {self.df['label'].value_counts().to_dict()}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        """
        Retorna:
            features: Tensor [287] com LULC features
            spatial_features: Tensor [3] com features espaciais para gate
            label: Tensor [1] com label (0=lento, 1=rápido)
        """
        row_data = self.df.iloc[idx]
        
        # Extrair features LULC (placeholder - usar extração real)
        # TODO: Implementar extração de features igual ao baseline
        # Por enquanto, usar features aleatórias para testar arquitetura
        features = self._extract_lulc_features(row_data)
        
        # Extrair spatial features para gate
        spatial_features = self._extract_spatial_features(row_data)
        
        # Label
        label = torch.tensor([row_data['label']], dtype=torch.float32)
        
        if self.transform:
            features = self.transform(features)
        
        return features, spatial_features, label
    
    def _extract_lulc_features(self, row_data):
        """
        Extrai features LULC para o pixel.
        
        TODO: Implementar extração real usando:
        - LULC time series (T-39 até T-1)
        - Spatial patch (5×7×7)
        - Auxiliary features (T, lat, lon)
        - Class embeddings (aprendidos)
        
        Por enquanto: features aleatórias (placeholder)
        """
        # PLACEHOLDER: Substituir por extração real!
        features = torch.randn(287)
        return features
    
    def _extract_spatial_features(self, row_data):
        """
        Extrai spatial features para gate network.
        
        Features:
        - is_cluster_conversion: 1 se conversão em cluster, 0 se disperso
        - is_cluster_use: 1 se uso final em cluster, 0 se disperso
        - consolidation_level: 0-3 (ordinal)
        
        Se dataset tiver essas colunas, usar. Senão, inferir do pattern_code.
        """
        # Tentar usar colunas se existirem
        if 'is_cluster_conversion' in row_data:
            is_cluster_conversion = float(row_data['is_cluster_conversion'])
            is_cluster_use = float(row_data['is_cluster_use'])
            consolidation_level = float(row_data['consolidation_level'])
        else:
            # Inferir do pattern_code se disponível
            if 'pattern_code' in row_data:
                pattern = str(row_data['pattern_code'])
                is_cluster_conversion = 1.0 if pattern.startswith('CC') else 0.0
                is_cluster_use = 1.0 if 'C' in pattern[-2:] else 0.0
                
                # Consolidation level baseado em padrão
                if pattern == 'CCAC':
                    consolidation_level = 3.0
                elif pattern.startswith('CC'):
                    consolidation_level = 2.0
                elif pattern.startswith('CD'):
                    consolidation_level = 1.0
                else:
                    consolidation_level = 0.0
            else:
                # Fallback: usar valores neutros
                is_cluster_conversion = 0.5
                is_cluster_use = 0.5
                consolidation_level = 1.5
        
        spatial_features = torch.tensor([
            is_cluster_conversion,
            is_cluster_use,
            consolidation_level
        ], dtype=torch.float32)
        
        return spatial_features


# ============================================================
# TRAINING FUNCTIONS
# ============================================================

def set_seed(seed):
    """Fixa random seed para reprodutibilidade."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)


def train_epoch(model, loader, criterion, optimizer, device):
    """Treina por uma epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training")
    for features, spatial_features, labels in pbar:
        features = features.to(device)
        spatial_features = spatial_features.to(device)
        labels = labels.to(device)
        
        # Forward
        predictions, gate_weights, head_outputs = model(features, spatial_features)
        loss = criterion(predictions, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Métricas
        total_loss += loss.item()
        pred_class = (predictions > 0.5).float()
        correct += (pred_class == labels).sum().item()
        total += labels.size(0)
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })
    
    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    """Avalia modelo."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    all_predictions = []
    all_labels = []
    all_gate_weights = []
    
    with torch.no_grad():
        for features, spatial_features, labels in tqdm(loader, desc="Evaluating"):
            features = features.to(device)
            spatial_features = spatial_features.to(device)
            labels = labels.to(device)
            
            # Forward
            predictions, gate_weights, head_outputs = model(features, spatial_features)
            loss = criterion(predictions, labels)
            
            # Métricas
            total_loss += loss.item()
            pred_class = (predictions > 0.5).float()
            correct += (pred_class == labels).sum().item()
            total += labels.size(0)
            
            # Armazenar para métricas detalhadas
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_gate_weights.extend(gate_weights.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    
    # Calcular métricas adicionais
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    
    predictions_np = np.array(all_predictions).flatten()
    labels_np = np.array(all_labels).flatten()
    pred_class_np = (predictions_np > 0.5).astype(int)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_np, pred_class_np, average='binary'
    )
    
    try:
        auc = roc_auc_score(labels_np, predictions_np)
    except:
        auc = 0.0
    
    metrics = {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }
    
    return metrics, all_gate_weights


def train_model(config):
    """Training loop completo."""
    
    print("="*70)
    print("MULTI-HEAD SPATIAL MODEL TRAINING")
    print("="*70)
    
    # Set seed
    set_seed(config['seed'])
    
    # Device
    device = torch.device(config['device'])
    print(f"\nDevice: {device}")
    
    # Criar output directory
    output_dir = Path(config['base_dir']) / config['output_dir']
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # Salvar config
    with open(output_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)
    
    # ==================== LOAD DATA ====================
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)
    
    dataset_path = Path(config['dataset_path'])
    
    if config['use_existing_splits']:
        # Usar splits existentes (do baseline)
        base_dir = Path(config['base_dir'])
        
        # Carregar indices
        train_indices = pd.read_csv(base_dir / config['train_csv'])['index'].values
        val_indices = pd.read_csv(base_dir / config['val_csv'])['index'].values
        test_indices = pd.read_csv(base_dir / config['test_csv'])['index'].values
        
        print(f"\nUsando splits existentes:")
        print(f"  Train indices: {len(train_indices):,}")
        print(f"  Val indices: {len(val_indices):,}")
        print(f"  Test indices: {len(test_indices):,}")
        
        # Criar datasets
        print(f"\nCarregando dataset: {dataset_path.name}")
        train_dataset = LULCMultiHeadDataset(dataset_path, config['lulc_dir'], indices=train_indices)
        val_dataset = LULCMultiHeadDataset(dataset_path, config['lulc_dir'], indices=val_indices)
        test_dataset = LULCMultiHeadDataset(dataset_path, config['lulc_dir'], indices=test_indices)
    
    else:
        # Split aleatório simples
        full_dataset = LULCMultiHeadDataset(dataset_path, config['lulc_dir'])
        
        train_size = int(0.7 * len(full_dataset))
        val_size = int(0.15 * len(full_dataset))
        test_size = len(full_dataset) - train_size - val_size
        
        train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
            full_dataset, [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(config['seed'])
        )
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=0  # Windows compatibility
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=0
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=0
    )
    
    # ==================== MODEL ====================
    print("\n" + "="*70)
    print("INITIALIZING MODEL")
    print("="*70)
    
    model = MultiHeadSpatialModel(
        input_dim=config['input_dim'],
        hidden_dim=config['hidden_dim'],
        encoder_dim=config['encoder_dim'],
        head_hidden_dim=config['head_hidden_dim'],
        dropout=config['dropout'],
        use_batch_norm=config['use_batch_norm']
    ).to(device)
    
    # Print summary
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    
    # Loss e optimizer
    criterion = nn.BCELoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5, verbose=True
    )
    
    # ==================== TRAINING ====================
    print("\n" + "="*70)
    print("TRAINING")
    print("="*70)
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_metrics': []}
    
    for epoch in range(config['n_epochs']):
        print(f"\nEpoch {epoch+1}/{config['n_epochs']}")
        print("-" * 70)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_metrics, val_gate_weights = evaluate(model, val_loader, criterion, device)
        
        # Scheduler
        scheduler.step(val_metrics['loss'])
        
        # History
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_metrics'].append(val_metrics)
        
        # Print
        print(f"\nTrain - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"Val   - Loss: {val_metrics['loss']:.4f}, Acc: {val_metrics['accuracy']:.4f}, "
              f"F1: {val_metrics['f1']:.4f}, AUC: {val_metrics['auc']:.4f}")
        
        # Save best model
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            patience_counter = 0
            
            if config['save_checkpoints']:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_metrics': val_metrics,
                    'config': config
                }, output_dir / 'best_model.pth')
                print("  → Best model saved!")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping after {epoch+1} epochs")
            break
    
    # ==================== FINAL EVALUATION ====================
    print("\n" + "="*70)
    print("FINAL EVALUATION ON TEST SET")
    print("="*70)
    
    # Load best model
    checkpoint = torch.load(output_dir / 'best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Evaluate on test
    test_metrics, test_gate_weights = evaluate(model, test_loader, criterion, device)
    
    print(f"\nTest Metrics:")
    print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"  Precision: {test_metrics['precision']:.4f}")
    print(f"  Recall:    {test_metrics['recall']:.4f}")
    print(f"  F1 Score:  {test_metrics['f1']:.4f}")
    print(f"  ROC AUC:   {test_metrics['auc']:.4f}")
    
    # Análise de gate weights
    gate_weights_np = np.array(test_gate_weights)
    print(f"\nGate Weights Analysis (Test Set):")
    print(f"  Disperso head - Mean: {gate_weights_np[:,0].mean():.3f}, Std: {gate_weights_np[:,0].std():.3f}")
    print(f"  Cluster head  - Mean: {gate_weights_np[:,1].mean():.3f}, Std: {gate_weights_np[:,1].std():.3f}")
    
    # Save results
    results = {
        'test_metrics': test_metrics,
        'history': history,
        'best_epoch': checkpoint['epoch'],
        'gate_weights_stats': {
            'disperso_mean': float(gate_weights_np[:,0].mean()),
            'disperso_std': float(gate_weights_np[:,0].std()),
            'cluster_mean': float(gate_weights_np[:,1].mean()),
            'cluster_std': float(gate_weights_np[:,1].std())
        }
    }
    
    with open(output_dir / 'results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✅ Training concluído!")
    print(f"   Resultados salvos em: {output_dir}")
    
    return model, results


if __name__ == "__main__":
    results = train_model(CONFIG)

MULTI-HEAD SPATIAL MODEL TRAINING

Device: cuda

LOADING DATA


KeyError: 'index'